In [5]:
import pygame
import random
import math
from collections import deque
import heapq


GOAL = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5),
)


def flatten(state):
    return [x for row in state for x in row]


def inversions(seq):
    vals = [x for x in seq if x != 0]
    inv = 0
    for i in range(len(vals)):
        for j in range(i + 1, len(vals)):
            if vals[i] > vals[j]:
                inv += 1
    return inv


def is_solvable(start, goal=GOAL):
    return (inversions(flatten(start)) % 2) == (inversions(flatten(goal)) % 2)


def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    raise ValueError("Board does not contain 0")


def swap(state, i1, j1, i2, j2):
    s = [list(r) for r in state]
    s[i1][j1], s[i2][j2] = s[i2][j2], s[i1][j1]
    return tuple(tuple(r) for r in s)


def get_neighbors(state):
    i, j = find_zero(state)
    moves = (
        ("UP", -1, 0),
        ("DOWN", 1, 0),
        ("LEFT", 0, -1),
        ("RIGHT", 0, 1),
    )
    neighbors = []
    for action, di, dj in moves:
        ni, nj = i + di, j + dj
        if 0 <= ni < 3 and 0 <= nj < 3:
            neighbors.append((swap(state, i, j, ni, nj), action))
    return neighbors


def build_path(parent, end_state):
    path = []
    s = end_state
    while parent[s][0] is not None:
        path.append((parent[s][1], s))
        s = parent[s][0]
    path.reverse()
    return path


def bfs(start):
    if start == GOAL:
        return []
    if not is_solvable(start):
        return None

    parent = {start: (None, None)}
    queue = deque([start])
    visited = {start}

    while queue:
        current = queue.popleft()
        if current == GOAL:
            return build_path(parent, current)

        for child, action in get_neighbors(current):
            if child not in visited:
                visited.add(child)
                parent[child] = (current, action)
                queue.append(child)

    return None


def dfs_early_exit(start):
    if start == GOAL:
        return []
    if not is_solvable(start):
        return None

    parent = {start: (None, None)}
    stack = deque([start])
    visited = {start}

    while stack:
        current = stack.pop()

        for child, action in get_neighbors(current):
            if child in visited:
                continue

            parent[child] = (current, action)
            if child == GOAL:
                return build_path(parent, child)

            visited.add(child)
            stack.append(child)

    return None


def _dls(node, depth, limit, parent, seen):
    if node == GOAL:
        return True
    if depth == limit:
        return False

    for child, action in get_neighbors(node):
        if child in seen:
            continue
        seen.add(child)
        parent[child] = (node, action)
        if _dls(child, depth + 1, limit, parent, seen):
            return True
        seen.remove(child)
    return False


def iddfs(start, max_depth=50):
    if start == GOAL:
        return []
    if not is_solvable(start):
        return None

    for limit in range(max_depth + 1):
        parent = {start: (None, None)}
        seen = {start}
        found = _dls(start, 0, limit, parent, seen)
        if found and GOAL in parent:
            return build_path(parent, GOAL)
    return None

def misplaced_tiles(state):
    count = 0

    for i in range(3):
        for j in range(3):
            if state[i][j] != 0 and state[i][j] != GOAL[i][j]:
                count += 1

    return count
 
def ucs(start):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    parent = {start: (None, None)}
    cost_so_far = {start: 0}

    frontier = []
    counter = 0

    heapq.heappush(frontier, (0, counter, start))

    explored = set()

    while frontier:
        current_cost, _, current = heapq.heappop(frontier)

        if current in explored:
            continue

        if current == GOAL:
            path = []
            s = current

            while parent[s][0] is not None:
                path.append((parent[s][1], s))
                s = parent[s][0]

            path.reverse()

            print("Total cost:", current_cost)
            return path

        explored.add(current)

        for child, action in get_neighbors(current):
            if child in explored:
                continue


            step_cost = misplaced_tiles(child)

            new_cost = current_cost + step_cost

            if child not in cost_so_far or new_cost < cost_so_far[child]:
                cost_so_far[child] = new_cost
                parent[child] = (current, action)

                counter += 1
                heapq.heappush(frontier, (new_cost, counter, child))

    return None

def manhattan_distance(state):
    total = 0

    for i in range(3):
        for j in range(3):
            val = state[i][j]

            if val != 0:
                for gi in range(3):
                    for gj in range(3):
                        if GOAL[gi][gj] == val:
                            total += abs(i - gi) + abs(j - gj)

    return total

def greedy_search(start):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    parent = {start: (None, None)}
    frontier = []
    reached = set()

    counter = 0
    heapq.heappush(frontier, (manhattan_distance(start), counter, start))

    while frontier:
        h_value, _, current = heapq.heappop(frontier)

        if current in reached:
            continue

        if current == GOAL:
            print("Greedy total heuristic at goal:", h_value)
            return build_path(parent, current)

        reached.add(current)

        for child, action in get_neighbors(current):
            if child not in reached:
                parent[child] = (current, action)

                counter += 1
                h_child = manhattan_distance(child)

                heapq.heappush(frontier, (h_child, counter, child))

    return None
def inversion_score(state, start=None):
    if start is not None and state == start:
        return 0

    return inversions(flatten(state))


def astar(start):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    parent = {start: (None, None)}
    frontier = []
    reached = set()
    best_f = {}

    counter = 0

    g_start = inversion_score(start, start)
    h_start = misplaced_tiles(start)
    f_start = g_start + h_start

    best_f[start] = f_start
    heapq.heappush(frontier, (f_start, counter, start))

    while frontier:
        current_f, _, current = heapq.heappop(frontier)

        if current in reached:
            continue

        if current == GOAL:
            print("A* Custom found goal")
            print("g(n) inversion =", inversion_score(current, start))
            print("h(n) misplaced =", misplaced_tiles(current))
            print("f(n) =", current_f)
            return build_path(parent, current)

        reached.add(current)

        for child, action in get_neighbors(current):
            if child in reached:
                continue

            g_child = inversion_score(child, start)
            h_child = misplaced_tiles(child)
            f_child = g_child + h_child

            if child not in best_f or f_child < best_f[child]:
                best_f[child] = f_child
                parent[child] = (current, action)

                counter += 1
                heapq.heappush(frontier, (f_child, counter, child))

    return None

def ida_f(state):
    return manhattan_distance(state) + misplaced_tiles(state)


def ida_star(start, max_depth=40):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    bound = ida_f(start)
    path = []
    path_set = {start}

    while True:
        print("IDA* bound =", bound)

        result, solution = ida_search(
            start,
            bound,
            path,
            path_set,
            depth=0,
            max_depth=max_depth
        )

        if result == "FOUND":
            return solution

        if result == float("inf"):
            return None

        bound = result


def ida_search(current, bound, path, path_set, depth, max_depth):
    f_current = ida_f(current)

    if f_current > bound:
        return f_current, None

    if depth > max_depth:
        return float("inf"), None

    if current == GOAL:
        return "FOUND", path.copy()

    min_bound = float("inf")

    neighbors = get_neighbors(current)
    neighbors.sort(key=lambda item: ida_f(item[0]))

    for child, action in neighbors:
        if child in path_set:
            continue

        path.append((action, child))
        path_set.add(child)

        result, solution = ida_search(
            child,
            bound,
            path,
            path_set,
            depth + 1,
            max_depth
        )

        if result == "FOUND":
            return "FOUND", solution

        if result < min_bound:
            min_bound = result

        path.pop()
        path_set.remove(child)

    return min_bound, None


def simple_hill_climbing(start):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    current = start
    path = []
    visited = {start}

    while True:
        current_h = manhattan_distance(current)
        found_better = False

        for child, action in get_neighbors(current):
            if child in visited:
                continue

            child_h = manhattan_distance(child)

            if child_h < current_h:
                path.append((action, child))
                current = child
                visited.add(current)
                found_better = True
                break

        if current == GOAL:
            print("Hill Climbing found goal")
            print("h(n) Manhattan =", manhattan_distance(current))
            return path

        if not found_better:
            print("Hill Climbing stopped at local optimum")
            print("Current h(n) =", manhattan_distance(current))
            return None


def steepest_hill_climbing(start):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    current = start
    path = []
    visited = {start}

    while True:
        current_h = manhattan_distance(current)
        best_child = None
        best_action = None
        best_h = float("inf")

        for child, action in get_neighbors(current):
            if child in visited:
                continue

            child_h = manhattan_distance(child)
            print("Current h =", current_h, "| Action =", action, "| Child h =", child_h)

            if child_h < best_h:
                best_h = child_h
                best_child = child
                best_action = action

        if best_child is None:
            print("Steepest Hill Climbing stopped: no available neighbor")
            return None

        if best_h < current_h:
            print("Choose action:", best_action, "with h =", best_h)
            path.append((best_action, best_child))
            current = best_child
            visited.add(current)

            if current == GOAL:
                print("Steepest Hill Climbing found goal")
                print("h(n) Manhattan =", manhattan_distance(current))
                return path
        else:
            print("Steepest Hill Climbing stopped at local optimum")
            print("Current h(n) =", current_h)
            print("Best neighbor h(n) =", best_h)
            return None


def stochastic_hill_climbing(start, max_steps=1000):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    current = start
    path = []

    for step in range(max_steps):
        current_h = manhattan_distance(current)
        print("\\nStep:", step)
        print("Current h =", current_h)
        print("Current state =", current)

        better_neighbors = []

        for child, action in get_neighbors(current):
            child_h = manhattan_distance(child)
            print("Action =", action, "| Child h =", child_h)

            if child_h < current_h:
                better_neighbors.append((child, action, child_h))

        if not better_neighbors:
            print("Stochastic Hill Climbing stopped at local optimum")
            print("Current h =", current_h)
            return None

        child, action, child_h = random.choice(better_neighbors)
        print("Randomly selected action:", action)
        print("Selected h =", child_h)

        path.append((action, child))
        current = child

        if current == GOAL:
            print("Stochastic Hill Climbing found goal")
            print("Number of moves =", len(path))
            return path

    print("Stochastic Hill Climbing reached max steps")
    return None


def local_beam_search(start, k=2, max_iterations=1000):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    current_state_set = [start]
    parent = {start: (None, None)}
    visited = {start}

    for iteration in range(max_iterations):
        print("Iteration:", iteration)
        print("Current beam:")

        for state in current_state_set:
            print("h =", manhattan_distance(state))
            for row in state:
                print(row)
            print()

        neighbor_states = []

        for current in current_state_set:
            for child, action in get_neighbors(current):
                if child in visited:
                    continue

                child_h = manhattan_distance(child)
                print("Action =", action, "| h =", child_h, "| Child =", child)

                parent[child] = (current, action)
                visited.add(child)

                if child == GOAL:
                    print("Local Beam Search found Goal")
                    return build_path(parent, child)

                neighbor_states.append((child_h, child))

        if not neighbor_states:
            print("Local Beam Search stopped: no neighbor states")
            return None

        neighbor_states.sort(key=lambda item: item[0])
        current_state_set = [state for _, state in neighbor_states[:k]]

    print("Local Beam Search reached max iterations")
    return None


def random_walk_restart(start, steps):
    current = start
    prefix_path = []
    previous_state = None

    for _ in range(steps):
        neighbors = get_neighbors(current)
        choices = [
            (child, action)
            for child, action in neighbors
            if child != previous_state
        ]

        if not choices:
            choices = neighbors

        child, action = random.choice(choices)
        prefix_path.append((action, child))
        previous_state = current
        current = child

        if current == GOAL:
            break

    return current, prefix_path


def random_restart_hill_climbing(start, max_restarts=100, max_random_steps=30):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    for attempt in range(max_restarts + 1):
        print("Random Restart attempt:", attempt)

        if attempt == 0:
            restart_state = start
            prefix_path = []
            random_steps = 0
        else:
            random_steps = random.randint(1, max_random_steps)
            restart_state, prefix_path = random_walk_restart(start, random_steps)

        print("Random walk steps:", random_steps)
        print("Restart state:", restart_state)
        print("Restart h(n):", manhattan_distance(restart_state))

        if restart_state == GOAL:
            print("Goal found during random walk")
            print("Total moves:", len(prefix_path))
            return prefix_path

        hill_path = steepest_hill_climbing(restart_state)

        if hill_path is not None:
            full_path = prefix_path + hill_path
            print("Random-Restart Hill Climbing found Goal")
            print("Successful attempt:", attempt)
            print("Prefix moves:", len(prefix_path))
            print("Hill Climbing moves:", len(hill_path))
            print("Total moves:", len(full_path))
            return full_path

    print("Maximum restart attempts reached:", max_restarts)
    return None


def simulated_annealing(start, initial_temperature=8.0, cooling_rate=0.995, min_temperature=0.01, max_steps=5000):
    if start == GOAL:
        return []

    if not is_solvable(start):
        return None

    current = start
    previous_state = None
    path = []
    best_state = start
    best_h = manhattan_distance(start)
    temperature = initial_temperature

    for step in range(max_steps):
        current_h = manhattan_distance(current)

        if current == GOAL:
            print("Simulated Annealing found goal")
            print("Number of moves =", len(path))
            return path

        if temperature < min_temperature:
            break

        neighbors = get_neighbors(current)
        choices = [(child, action) for child, action in neighbors if child != previous_state]

        if not choices:
            choices = neighbors

        child, action = random.choice(choices)
        child_h = manhattan_distance(child)
        delta = current_h - child_h

        if delta >= 0:
            accept = True
            probability = 1.0
        else:
            probability = math.exp(delta / temperature)
            accept = random.random() < probability

        print(
            "Step:", step,
            "| Temp =", round(temperature, 4),
            "| Action =", action,
            "| Current h =", current_h,
            "| Child h =", child_h,
            "| Accept p =", round(probability, 4),
            "| Accepted =", accept,
        )

        if accept:
            path.append((action, child))
            previous_state = current
            current = child

            if child_h < best_h:
                best_h = child_h
                best_state = child

        temperature *= cooling_rate

    print("Simulated Annealing stopped before reaching goal")
    print("Best h(n) =", best_h)
    print("Best state =", best_state)
    return None


pygame.init()
pygame.display.set_caption("8 Puzzle Search Visualizer")


SCREEN_W, SCREEN_H = 1600, 900
screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))
clock = pygame.time.Clock()


FONT_TITLE = pygame.font.SysFont("Segoe UI", 38, bold=True)
FONT_HEAD = pygame.font.SysFont("Segoe UI", 22, bold=True)
FONT_BODY = pygame.font.SysFont("Segoe UI", 17)
FONT_SMALL = pygame.font.SysFont("Segoe UI", 14)
FONT_TINY = pygame.font.SysFont("Segoe UI", 12)
FONT_TILE = pygame.font.SysFont("Segoe UI", 82, bold=True)
FONT_BADGE = pygame.font.SysFont("Segoe UI", 14, bold=True)
FONT_METRIC = pygame.font.SysFont("Segoe UI", 19, bold=True)


BG = (25, 20, 36)
BG_2 = (35, 28, 52)
CARD = (45, 36, 66)
CARD_2 = (58, 46, 84)
BORDER = (116, 88, 166)
TEXT = (250, 245, 255)
MUTED = (190, 175, 215)
ACCENT = (216, 180, 254)
ACCENT_2 = (147, 51, 234)
SUCCESS = (86, 211, 100)
WARNING = (251, 191, 36)
DANGER = (248, 113, 113)
EMPTY_TILE = (32, 25, 48)
TILE_BG = (168, 85, 247)
TILE_BG_HOVER = (192, 132, 252)



MARGIN = 28
CONTENT_TOP = 92
CONTENT_H = 780

BOARD_CARD = pygame.Rect(28, CONTENT_TOP, 560, CONTENT_H)
BOARD_AREA = pygame.Rect(58, 160, 500, 500)
TILE_GAP = 12
TILE_SIZE = (BOARD_AREA.w - TILE_GAP * 4) // 3

MID_X = 616
PANEL_W = 480
CONTROL_PANEL = pygame.Rect(MID_X, CONTENT_TOP, PANEL_W, 390)
STATS_PANEL = pygame.Rect(MID_X, 502, PANEL_W, 130)
MOVES_PANEL = pygame.Rect(MID_X, 652, PANEL_W, 220)
STATES_PANEL = pygame.Rect(1124, CONTENT_TOP, 448, CONTENT_H)

ACTION_GAP = 12
ACTION_W = (CONTROL_PANEL.w - 36 - ACTION_GAP) // 2
btn_start = pygame.Rect(CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 62, ACTION_W, 44)
btn_shuffle = pygame.Rect(btn_start.right + ACTION_GAP, CONTROL_PANEL.y + 62, ACTION_W, 44)

algorithms = [
    "BFS", "DFS", "IDDFS", "UCS", "Greedy", "A*", "IDA*",
    "Hill Climb", "Steepest Hill Climb", "Stochastic Hill Climb",
    "Simulated Annealing", "Local Beam Search", "Random-Restart Hill Climb"
]
ALG_COLS = 2
ALG_GAP_X = 12
ALG_GAP_Y = 6
ALG_BTN_W = (CONTROL_PANEL.w - 36 - ALG_GAP_X) // ALG_COLS
ALG_BTN_H = 28
alg_buttons = [
    pygame.Rect(
        CONTROL_PANEL.x + 18 + (i % ALG_COLS) * (ALG_BTN_W + ALG_GAP_X),
        CONTROL_PANEL.y + 150 + (i // ALG_COLS) * (ALG_BTN_H + ALG_GAP_Y),
        ALG_BTN_W,
        ALG_BTN_H,
    )
    for i in range(len(algorithms))
]

start_board = ((1, 2, 3), (8, 4, 0), (7, 6, 5))
board = [list(r) for r in start_board]
orig_board = [row[:] for row in board]

selected_alg = 0
solution_path = None
solution_states = []
moves_used = []
status_message = "Ready. Choose an algorithm and press Start."
current_step_index = 0

state_scroll_y = 0
moves_scroll_y = 0

active_tab = 0
APP_TABS = ["Puzzle Search", "Belief Search", "CSP Search", "AND-OR Search"]
TAB_W = 198
TAB_H = 32
tab_buttons = [
    pygame.Rect(MARGIN + i * (TAB_W + 10), 56, TAB_W, TAB_H)
    for i in range(len(APP_TABS))
]


dragging = False
drag_tile = None         
drag_value = None         
drag_offset = (0, 0)     
drag_pos = (0, 0)        

MOVE_LABELS = {
    "UP": "Move Up",
    "DOWN": "Move Down",
    "LEFT": "Move Left",
    "RIGHT": "Move Right",
}


BELIEF_START_A = (
    (1, 0, 3),
    (8, 2, 4),
    (7, 6, 5),
)
BELIEF_START_B = (
    (1, 2, 3),
    (8, 4, 0),
    (7, 6, 5),
)
BELIEF_START = (BELIEF_START_A, BELIEF_START_B)
BELIEF_GOAL_ONE = GOAL
BELIEF_GOAL_TWO = (
    (0, 1, 3),
    (8, 2, 4),
    (7, 6, 5),
)
BELIEF_ACTIONS = ("UP", "DOWN", "LEFT", "RIGHT")
BELIEF_MOVE_DELTAS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}
BELIEF_OBSERVATION_NAMES = ("Top row", "Middle row", "Bottom row")
BELIEF_MODES = ["No Observation", "Partial Observation"]
BELIEF_GOAL_MODES = ["1 Goal", "2 Goals"]

BELIEF_WORLD_AREA = pygame.Rect(78, 188, 460, 368)
BELIEF_RESULT_PANEL = STATS_PANEL
BELIEF_PLAN_PANEL = MOVES_PANEL
BELIEF_TREE_PANEL = STATES_PANEL

belief_mode = 0
belief_goal_mode = 0
belief_status = "Ready. Choose a belief problem and press Solve."
belief_solution_path = []
belief_state_history = [BELIEF_START]
belief_plan_lines = []
belief_plan_tree = None
belief_solution_cost = 0
belief_scroll_y = 0
belief_tree_scroll_y = 0

BELIEF_MODE_BUTTONS = [
    pygame.Rect(CONTROL_PANEL.x + 18 + i * 222, CONTROL_PANEL.y + 92, 210, 38)
    for i in range(len(BELIEF_MODES))
]
BELIEF_GOAL_BUTTONS = [
    pygame.Rect(CONTROL_PANEL.x + 18 + i * 222, CONTROL_PANEL.y + 172, 210, 38)
    for i in range(len(BELIEF_GOAL_MODES))
]
btn_belief_solve = pygame.Rect(CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 272, ACTION_W, 44)
btn_belief_reset = pygame.Rect(btn_belief_solve.right + ACTION_GAP, CONTROL_PANEL.y + 272, ACTION_W, 44)


CSP_BACKTRACK_VARIABLES = [f"X{i}" for i in range(1, 10)]
CSP_BACKTRACK_VALUES = [1, 2, 3, 4, 5, 6, 7, 8]
CSP_BACKTRACK_RULES = {f"X{i}": i for i in range(1, 9)}
CSP_BACKTRACK_RULES["X9"] = 0
CSP_BACKTRACK_DOMAINS = {var: CSP_BACKTRACK_VALUES[:] for var in CSP_BACKTRACK_VARIABLES}
CSP_BACKTRACK_DOMAINS["X9"] = [0]
CSP_BACKTRACK_NODE_POS = {
    "X1": (95, 85),
    "X2": (230, 85),
    "X3": (365, 85),
    "X4": (95, 220),
    "X5": (230, 220),
    "X6": (365, 220),
    "X7": (95, 355),
    "X8": (230, 355),
    "X9": (365, 355),
}
CSP_BACKTRACK_VALUE_COLORS = {
    0: (31, 39, 56),
    1: (96, 165, 250),
    2: (86, 211, 100),
    3: (251, 191, 36),
    4: (248, 113, 113),
    5: (167, 139, 250),
    6: (45, 212, 191),
    7: (251, 146, 60),
    8: (244, 114, 182),
}

CSP_COLOR_VARIABLES = ["A", "B", "C", "D", "E"]
CSP_COLOR_VALUES = ["Đỏ", "Xanh", "Vàng", "Trắng"]
CSP_COLOR_DOMAINS = {var: CSP_COLOR_VALUES[:] for var in CSP_COLOR_VARIABLES}
CSP_COLOR_EDGES = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("B", "D"),
    ("C", "D"),
    ("C", "E"),
    ("D", "E"),
]
CSP_COLOR_NODE_POS = {
    "A": (225, 80),
    "B": (105, 235),
    "C": (345, 235),
    "D": (160, 405),
    "E": (300, 405),
}
CSP_COLOR_VALUE_COLORS = {
    "Đỏ": (248, 113, 113),
    "Xanh": (96, 165, 250),
    "Vàng": (251, 191, 36),
    "Trắng": (235, 240, 245),
}

CSP_MAP_VARIABLES = ["A", "B", "C", "D"]
CSP_MAP_VALUES = ["Đỏ", "Xanh", "Vàng"]
CSP_MAP_DOMAINS = {var: CSP_MAP_VALUES[:] for var in CSP_MAP_VARIABLES}
CSP_MAP_EDGES = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("B", "D"),
    ("C", "D"),
]
CSP_MAP_NODE_POS = {
    "A": (225, 90),
    "B": (105, 245),
    "C": (345, 245),
    "D": (225, 410),
}
CSP_MAP_VALUE_COLORS = {
    "Đỏ": (248, 113, 113),
    "Xanh": (96, 165, 250),
    "Vàng": (251, 191, 36),
}

CSP_ARC_VARIABLES = ["X", "Y", "Z"]
CSP_ARC_DOMAINS = {
    "X": [1, 2, 3],
    "Y": [1, 2, 3],
    "Z": [2, 3],
}
CSP_ARC_EDGES = [
    ("X", "Y"),
    ("Y", "Z"),
]
CSP_ARC_ARCS = [
    ("X", "Y"),
    ("Y", "X"),
    ("Y", "Z"),
    ("Z", "Y"),
]
CSP_ARC_NODE_POS = {
    "X": (105, 250),
    "Y": (225, 120),
    "Z": (345, 250),
}
CSP_MIN_CONFLICTS_MAX_STEPS = 80

CSP_MODES = ["Backtracking", "Forwardtrack", "AC-3", "Min-Conflicts"]
CSP_MODE_BUTTONS = [
    pygame.Rect(
        CONTROL_PANEL.x + 18 + (i % 2) * 222,
        CONTROL_PANEL.y + 92 + (i // 2) * 44,
        210,
        38,
    )
    for i in range(len(CSP_MODES))
]
btn_csp_solve = pygame.Rect(CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 310, ACTION_W, 44)
btn_csp_reset = pygame.Rect(btn_csp_solve.right + ACTION_GAP, CONTROL_PANEL.y + 310, ACTION_W, 44)

csp_mode = 0
csp_assignment = {}
csp_current_domains = {var: values[:] for var, values in CSP_BACKTRACK_DOMAINS.items()}
csp_steps = []
csp_status = "Ready. Choose a CSP algorithm and press Solve."
csp_nodes_visited = 0
csp_backtracks = 0
csp_steps_scroll_y = 0
csp_info_scroll_y = 0

ANDOR_START = start_board
ANDOR_MAX_DEPTH = 18
btn_andor_solve = pygame.Rect(CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 62, ACTION_W, 44)
btn_andor_shuffle = pygame.Rect(btn_andor_solve.right + ACTION_GAP, CONTROL_PANEL.y + 62, ACTION_W, 44)

andor_plan = None
andor_status = "Ready. Press Solve to run AND-OR graph search."
andor_steps = []
andor_history = [ANDOR_START]
andor_nodes_visited = 0
andor_and_nodes = 0
andor_policy_depth = 0
andor_steps_scroll_y = 0
andor_info_scroll_y = 0


def clamp(val, lo, hi):
    return max(lo, min(hi, val))


def board_to_tuple(b):
    return tuple(tuple(row) for row in b)


def active_belief_goals():
    goals = [BELIEF_GOAL_ONE]
    if belief_goal_mode == 1:
        goals.append(BELIEF_GOAL_TWO)
    return tuple(goals)


def board_key(state):
    return "".join(str(x) for row in state for x in row)


def belief_state_label(state):
    labels = {
        BELIEF_START_A: "S1",
        BELIEF_START_B: "S2",
        BELIEF_GOAL_ONE: "G1",
        BELIEF_GOAL_TWO: "G2",
    }
    if state in labels:
        return labels[state]

    zi, zj = find_zero(state)
    return f"h={manhattan_distance(state)}, blank=({zi + 1},{zj + 1})"


def format_belief(belief):
    states = sorted(belief, key=board_key)
    return "{" + ", ".join(belief_state_label(state) for state in states) + "}"


def attempted_puzzle_move(state, action):
    i, j = find_zero(state)
    di, dj = BELIEF_MOVE_DELTAS[action]
    ni, nj = i + di, j + dj

    if not (0 <= ni < 3 and 0 <= nj < 3):
        return state

    return swap(state, i, j, ni, nj)


def advance_belief(belief, action):
    return tuple(attempted_puzzle_move(state, action) for state in belief)


def belief_is_goal(belief, goals):
    return bool(belief) and any(all(state == goal for state in belief) for goal in goals)


def puzzle_manhattan_to_goal(state, goal_state):
    total = 0

    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val == 0:
                continue

            for gi in range(3):
                for gj in range(3):
                    if goal_state[gi][gj] == val:
                        total += abs(i - gi) + abs(j - gj)

    return total


def belief_manhattan_cost(belief, goals=None):
    active_goals = goals or active_belief_goals()
    return min(
        sum(puzzle_manhattan_to_goal(state, goal) for state in belief)
        for goal in active_goals
    )


def observation_for_state(state):
    row, _ = find_zero(state)
    return BELIEF_OBSERVATION_NAMES[row]


def observation_partition(belief):
    partitions = {}
    for state in belief:
        obs = observation_for_state(state)
        partitions.setdefault(obs, []).append(state)
    return {obs: tuple(cells) for obs, cells in partitions.items()}


def build_belief_path(parent, end_belief):
    path = []
    belief = end_belief
    while parent[belief][0] is not None:
        prev, action = parent[belief]
        path.append((action, belief))
        belief = prev
    path.reverse()
    return path


def no_observation_belief_search(start_belief, goals, max_depth=20):
    if belief_is_goal(start_belief, goals):
        return [], 0

    parent = {start_belief: (None, None)}
    best_cost = {start_belief: 0}
    frontier = []
    counter = 0
    heapq.heappush(frontier, (0, 0, counter, start_belief))

    while frontier:
        current_cost, current_depth, _, current = heapq.heappop(frontier)

        if current_cost != best_cost.get(current):
            continue

        if belief_is_goal(current, goals):
            return build_belief_path(parent, current), current_cost

        if current_depth >= max_depth:
            continue

        for action in BELIEF_ACTIONS:
            child = advance_belief(current, action)
            child_cost = current_cost + belief_manhattan_cost(child, goals)

            if child not in best_cost or child_cost < best_cost[child]:
                parent[child] = (current, action)
                best_cost[child] = child_cost

                counter += 1
                heapq.heappush(frontier, (child_cost, current_depth + 1, counter, child))

    return None


def partial_observation_belief_search(start_belief, goals, max_depth=12):
    best_plan = None
    best_cost = float("inf")

    for target_goal in goals:
        shared_goal = (target_goal,)

        for depth in range(max_depth + 1):
            plan = partial_observation_solve(start_belief, shared_goal, depth, {}, set())
            if plan is not None and plan["cost"] < best_cost:
                plan["target_goal"] = target_goal
                best_plan = plan
                best_cost = plan["cost"]

    return best_plan


def partial_observation_solve(belief, goals, depth, memo, visiting):
    if belief_is_goal(belief, goals):
        return {"type": "done", "belief": belief, "cost": 0}

    if depth == 0:
        return None

    key = (belief, depth)
    if key in memo:
        return memo[key]

    if belief in visiting:
        return None

    visiting.add(belief)
    best_plan = None
    best_cost = float("inf")

    for action in BELIEF_ACTIONS:
        predicted = advance_belief(belief, action)
        branches = {}
        branch_cost = 0
        works_for_all_observations = True

        for obs, observed_belief in sorted(observation_partition(predicted).items()):
            subplan = partial_observation_solve(observed_belief, goals, depth - 1, memo, visiting)
            if subplan is None:
                works_for_all_observations = False
                break
            branches[obs] = {"belief": observed_belief, "plan": subplan}
            branch_cost += subplan["cost"]

        if works_for_all_observations:
            step_cost = belief_manhattan_cost(predicted, goals)
            total_cost = step_cost + branch_cost

            if total_cost < best_cost:
                best_cost = total_cost
                best_plan = {
                    "type": "branch",
                    "belief": belief,
                    "action": action,
                    "predicted": predicted,
                    "step_cost": step_cost,
                    "cost": total_cost,
                    "branches": branches,
                }

    visiting.remove(belief)
    memo[key] = best_plan
    return best_plan


def partial_world_trace(start_belief, plan):
    tracks = [{"state": state, "plan": plan} for state in start_belief]
    history = [tuple(track["state"] for track in tracks)]
    step_lines = []
    step = 1

    while any(track["plan"]["type"] != "done" for track in tracks):
        actions_this_step = []
        next_tracks = []

        for track in tracks:
            node = track["plan"]
            state = track["state"]

            if node["type"] == "done":
                next_tracks.append(track)
                continue

            action = node["action"]
            next_state = attempted_puzzle_move(state, action)
            obs = observation_for_state(next_state)
            branch = node["branches"].get(obs)
            actions_this_step.append(action)

            next_tracks.append({
                "state": next_state,
                "plan": branch["plan"] if branch else {"type": "done", "belief": (next_state,), "cost": 0},
            })

        unique_actions = []
        for action in actions_this_step:
            if action not in unique_actions:
                unique_actions.append(action)

        labels = [MOVE_LABELS.get(action, action) for action in unique_actions]
        step_lines.append(f"{step:02d}. {' / '.join(labels)}")

        tracks = next_tracks
        history.append(tuple(track["state"] for track in tracks))
        step += 1

        if step > 50:
            break

    return history, step_lines


def count_partial_plan_actions(plan):
    if plan is None or plan["type"] == "done":
        return 0
    return 1 + sum(count_partial_plan_actions(branch["plan"]) for branch in plan["branches"].values())


def reset_belief_result(message=None):
    global belief_solution_path, belief_state_history, belief_plan_lines, belief_plan_tree
    global belief_solution_cost
    global belief_status, belief_scroll_y, belief_tree_scroll_y

    belief_solution_path = []
    belief_state_history = [BELIEF_START]
    belief_plan_lines = []
    belief_plan_tree = None
    belief_solution_cost = 0
    belief_scroll_y = 0
    belief_tree_scroll_y = 0
    belief_status = message or "Ready. Choose a belief problem and press Solve."


def run_belief_search():
    global belief_solution_path, belief_state_history, belief_plan_lines, belief_plan_tree
    global belief_solution_cost
    global belief_status, belief_scroll_y, belief_tree_scroll_y

    goals = active_belief_goals()
    belief_scroll_y = 0
    belief_tree_scroll_y = 0

    if belief_mode == 0:
        result = no_observation_belief_search(BELIEF_START, goals)
        belief_plan_tree = None

        if result is None:
            belief_solution_path = []
            belief_state_history = [BELIEF_START]
            belief_solution_cost = 0
            belief_plan_lines = ["UCS did not find a conformant step sequence within the depth limit."]
            belief_status = "No Observation UCS did not find guaranteed steps."
            return

        path, total_cost = result
        belief_solution_path = path
        belief_state_history = [BELIEF_START] + [belief for _, belief in path]
        belief_solution_cost = total_cost
        belief_plan_lines = []
        for idx, (action, belief) in enumerate(path, start=1):
            label = MOVE_LABELS.get(action, action)
            belief_plan_lines.append(f"{idx:02d}. {label}")
        belief_status = f"No Observation solved by UCS with g(n)={total_cost}."
        return

    plan = partial_observation_belief_search(BELIEF_START, goals)
    belief_solution_path = []
    belief_plan_tree = plan

    if plan is None:
        belief_state_history = [BELIEF_START]
        belief_solution_cost = 0
        belief_plan_lines = ["UCS-style conditional search did not find conditional steps within the depth limit."]
        belief_status = "Partial Observation UCS did not find conditional steps."
        return

    belief_state_history, belief_plan_lines = partial_world_trace(BELIEF_START, plan)
    belief_solution_cost = plan["cost"]
    action_count = count_partial_plan_actions(plan)
    belief_status = f"Partial Observation solved by UCS-style steps with g(n)={plan['cost']}."


def handle_belief_click(mx, my):
    global belief_mode, belief_goal_mode

    if btn_belief_solve.collidepoint(mx, my):
        run_belief_search()
        return

    if btn_belief_reset.collidepoint(mx, my):
        reset_belief_result()
        return

    for idx, rect in enumerate(BELIEF_MODE_BUTTONS):
        if rect.collidepoint(mx, my):
            belief_mode = idx
            reset_belief_result(f"Selected {BELIEF_MODES[belief_mode]}.")
            return

    for idx, rect in enumerate(BELIEF_GOAL_BUTTONS):
        if rect.collidepoint(mx, my):
            belief_goal_mode = idx
            reset_belief_result(f"Selected {BELIEF_GOAL_MODES[belief_goal_mode]}.")
            return


def andor_actions(state):
    actions = []
    for child, action in get_neighbors(state):
        actions.append((manhattan_distance(child), action))
    actions.sort(key=lambda item: (item[0], item[1]))
    return [action for _, action in actions]


def andor_results(state, action):
    for child, move in get_neighbors(state):
        if move == action:
            return (child,)
    return (state,)


def andor_or_search(state, path, depth, stats):
    stats["or_nodes"] += 1

    if state == GOAL:
        return {"type": "done", "state": state}

    if state in path:
        stats["failures"] += 1
        return None

    if depth >= ANDOR_MAX_DEPTH:
        stats["failures"] += 1
        return None

    for action in andor_actions(state):
        result_states = andor_results(state, action)
        and_plan = andor_and_search(result_states, path + [state], depth + 1, stats)

        if and_plan is not None:
            return {
                "type": "action",
                "state": state,
                "action": action,
                "results": result_states,
                "plans": and_plan,
            }

    stats["failures"] += 1
    return None


def andor_and_search(states, path, depth, stats):
    stats["and_nodes"] += 1
    plans = {}

    for state in states:
        plan_s = andor_or_search(state, path, depth, stats)
        if plan_s is None:
            return None
        plans[state] = plan_s

    return plans


def andor_graph_search(start_state):
    stats = {"or_nodes": 0, "and_nodes": 0, "failures": 0}
    plan = andor_or_search(start_state, [], 0, stats)
    return plan, stats


def andor_plan_step_count(plan):
    if plan is None or plan["type"] == "done":
        return 0
    branch_depths = [andor_plan_step_count(child_plan) for child_plan in plan["plans"].values()]
    return 1 + (max(branch_depths) if branch_depths else 0)


def format_andor_plan(plan, depth=0, lines=None, limit=80):
    if lines is None:
        lines = []

    if len(lines) >= limit:
        return lines

    prefix = "  " * depth

    if plan is None:
        lines.append(prefix + "failure")
        return lines

    if plan["type"] == "done":
        lines.append(prefix + "Goal reached -> return empty plan")
        return lines

    action_label = MOVE_LABELS.get(plan["action"], plan["action"])
    lines.append(prefix + f"OR: at {belief_state_label(plan['state'])}, choose {action_label}")
    lines.append(prefix + f"AND: solve all {len(plan['results'])} result state(s)")

    for idx, result_state in enumerate(plan["results"], start=1):
        child = plan["plans"][result_state]
        lines.append(prefix + f"  result {idx}: {belief_state_label(result_state)}")
        format_andor_plan(child, depth + 2, lines, limit)

    return lines


def andor_policy_trace(plan):
    history = []
    steps = []
    current = plan
    guard = 0

    while current is not None and guard < 50:
        history.append(current["state"])

        if current["type"] == "done":
            break

        action = current["action"]
        action_label = MOVE_LABELS.get(action, action)
        result_state = min(current["results"], key=manhattan_distance)
        steps.append(f"{len(steps) + 1:02d}. {action_label} -> {belief_state_label(result_state)}")
        current = current["plans"][result_state]
        guard += 1

    return history, steps


def randomize_andor_start(min_steps=4, max_steps=12):
    current = GOAL
    previous_state = None
    steps = random.randint(min_steps, max_steps)

    for _ in range(steps):
        choices = [
            (child, action)
            for child, action in get_neighbors(current)
            if child != previous_state
        ]
        if not choices:
            choices = get_neighbors(current)

        child, _ = random.choice(choices)
        previous_state = current
        current = child

    if current == GOAL:
        return start_board

    return current


def reset_andor_result(message=None):
    global andor_plan, andor_status, andor_steps, andor_history
    global andor_nodes_visited, andor_and_nodes, andor_policy_depth
    global andor_steps_scroll_y, andor_info_scroll_y

    andor_plan = None
    andor_steps = []
    andor_history = [ANDOR_START]
    andor_nodes_visited = 0
    andor_and_nodes = 0
    andor_policy_depth = 0
    andor_steps_scroll_y = 0
    andor_info_scroll_y = 0
    andor_status = message or "Ready. Press Solve to run AND-OR graph search."


def shuffle_andor_start():
    global ANDOR_START

    ANDOR_START = randomize_andor_start()
    reset_andor_result("New AND-OR board generated. Press Solve.")


def run_andor_search():
    global andor_plan, andor_status, andor_steps, andor_history
    global andor_nodes_visited, andor_and_nodes, andor_policy_depth
    global andor_steps_scroll_y, andor_info_scroll_y

    andor_steps_scroll_y = 0
    andor_info_scroll_y = 0
    plan, stats = andor_graph_search(ANDOR_START)
    andor_plan = plan
    andor_nodes_visited = stats["or_nodes"]
    andor_and_nodes = stats["and_nodes"]

    if plan is None:
        andor_history = [ANDOR_START]
        andor_steps = ["AND-OR graph search returned failure within the depth limit."]
        andor_policy_depth = 0
        andor_status = f"AND-OR failed within depth {ANDOR_MAX_DEPTH}."
        return

    plan_lines = format_andor_plan(plan)
    andor_history, trace_lines = andor_policy_trace(plan)
    andor_steps = plan_lines + ["", "Policy branch"] + trace_lines
    andor_policy_depth = andor_plan_step_count(plan)
    andor_status = f"AND-OR found a conditional plan with depth {andor_policy_depth}."


def handle_andor_click(mx, my):
    if btn_andor_solve.collidepoint(mx, my):
        run_andor_search()
        return

    if btn_andor_shuffle.collidepoint(mx, my):
        shuffle_andor_start()
        return


def csp_active_variables():
    if csp_mode == 0:
        return CSP_BACKTRACK_VARIABLES
    if csp_mode == 1:
        return CSP_COLOR_VARIABLES
    if csp_mode == 2:
        return CSP_ARC_VARIABLES
    return CSP_MAP_VARIABLES


def csp_active_domains():
    if csp_mode == 0:
        return CSP_BACKTRACK_DOMAINS
    if csp_mode == 1:
        return CSP_COLOR_DOMAINS
    if csp_mode == 2:
        return CSP_ARC_DOMAINS
    return CSP_MAP_DOMAINS


def csp_neighbors(var):
    result = []
    if csp_mode == 0:
        return result

    if csp_mode == 1:
        edges = CSP_COLOR_EDGES
    elif csp_mode == 2:
        edges = CSP_ARC_EDGES
    else:
        edges = CSP_MAP_EDGES

    for left, right in edges:
        if left == var:
            result.append(right)
        elif right == var:
            result.append(left)
    return result


def csp_is_consistent(var, value, assignment):
    if csp_mode == 0:
        if value != CSP_BACKTRACK_RULES[var]:
            return False

        for assigned_value in assignment.values():
            if value != 0 and assigned_value == value:
                return False

        return True

    if csp_mode == 2:
        for neighbor in csp_neighbors(var):
            if neighbor not in assignment:
                continue
            if not csp_arc_is_allowed(var, value, neighbor, assignment[neighbor]):
                return False
    else:
        for neighbor in csp_neighbors(var):
            if assignment.get(neighbor) == value:
                return False

    return True


def csp_select_unassigned_variable(assignment):
    for var in csp_active_variables():
        if var not in assignment:
            return var
    return None


def csp_copy_domains(domains):
    return {var: values[:] for var, values in domains.items()}


def csp_color_prune_domains(assignment, domains, source_var, source_value, steps):
    for neighbor in csp_neighbors(source_var):
        if neighbor in assignment:
            continue

        if source_value not in domains[neighbor]:
            continue

        domains[neighbor].remove(source_value)
        steps.append(f"Forwardtrack: bỏ {source_value} khỏi {neighbor}")

        if not domains[neighbor]:
            steps.append(f"Fail: {neighbor} has empty domain")
            return False

    return True


def csp_format_assignment(assignment, variables=None):
    active_variables = variables or csp_active_variables()
    return ", ".join(f"{var}={assignment[var]}" for var in active_variables if var in assignment)


def csp_format_domains(domains, variables=None):
    active_variables = variables or csp_active_variables()
    return ", ".join(f"{var}={domains[var]}" for var in active_variables)


def csp_arc_is_allowed(xi, value_i, xj, value_j):
    if xi == "X" and xj == "Y":
        return value_i < value_j
    if xi == "Y" and xj == "X":
        return value_j < value_i
    if xi == "Y" and xj == "Z":
        return value_i < value_j
    if xi == "Z" and xj == "Y":
        return value_j < value_i
    return True


def csp_arc_label(xi, xj):
    if (xi, xj) in (("X", "Y"), ("Y", "X")):
        return "X < Y"
    if (xi, xj) in (("Y", "Z"), ("Z", "Y")):
        return "Y < Z"
    return "constraint"


def csp_remove_inconsistent_values(xi, xj, domains, steps):
    removed_values = []

    for value in domains[xi][:]:
        has_support = any(
            csp_arc_is_allowed(xi, value, xj, other)
            for other in domains[xj]
        )
        if not has_support:
            domains[xi].remove(value)
            removed_values.append(value)

    for value in removed_values:
        steps.append(f"Remove {xi}={value}: no support in {xj}")

    return len(removed_values)


def csp_ac3():
    domains = csp_copy_domains(CSP_ARC_DOMAINS)
    queue = deque(CSP_ARC_ARCS)
    steps = []
    arcs_checked = 0
    removed_count = 0

    steps.append(f"Initial domains: {csp_format_domains(domains, CSP_ARC_VARIABLES)}")
    steps.append("Constraints: X < Y, Y < Z")
    steps.append(f"Initial queue: {len(queue)} arcs")

    while queue:
        xi, xj = queue.popleft()
        arcs_checked += 1
        steps.append(f"Remove first: ({xi}, {xj})")

        removed = csp_remove_inconsistent_values(xi, xj, domains, steps)
        if not removed:
            steps.append(f"Keep {xi}: every value has support in {xj}")
            continue

        removed_count += removed

        if not domains[xi]:
            steps.append(f"Fail: {xi} has empty domain")
            return None, domains, steps, arcs_checked, removed_count

        steps.append(f"Domain changed: {xi} = {domains[xi]}")
        for xk in csp_neighbors(xi):
            if xk == xj:
                continue
            queue.append((xk, xi))
            steps.append(f"Add ({xk}, {xi}) back to queue")

    steps.append(f"Final domains: {csp_format_domains(domains, CSP_ARC_VARIABLES)}")
    steps.append("AC-3 finished")
    return {}, domains, steps, arcs_checked, removed_count


def csp_total_conflicts(assignment):
    return sum(
        1
        for left, right in CSP_MAP_EDGES
        if assignment.get(left) == assignment.get(right)
    )


def csp_conflicts_for_value(var, value, assignment):
    conflicts = 0

    for left, right in CSP_MAP_EDGES:
        if left == var and assignment.get(right) == value:
            conflicts += 1
        elif right == var and assignment.get(left) == value:
            conflicts += 1

    return conflicts


def csp_conflicted_variables(assignment):
    return [
        var
        for var in CSP_MAP_VARIABLES
        if csp_conflicts_for_value(var, assignment[var], assignment) > 0
    ]


def csp_min_conflicts(max_steps=CSP_MIN_CONFLICTS_MAX_STEPS):
    current = {
        var: random.choice(CSP_MAP_DOMAINS[var])
        for var in CSP_MAP_VARIABLES
    }
    steps = [f"Initial complete assignment: {csp_format_assignment(current, CSP_MAP_VARIABLES)}"]

    for step in range(1, max_steps + 1):
        total_conflicts = csp_total_conflicts(current)
        if total_conflicts == 0:
            steps.append(f"Solution found at step {step - 1}")
            return current, steps, step - 1, 0

        conflicted = csp_conflicted_variables(current)
        var = random.choice(conflicted)
        scored_values = [
            (csp_conflicts_for_value(var, value, current), value)
            for value in CSP_MAP_DOMAINS[var]
        ]
        best_conflict = min(score for score, _ in scored_values)
        best_values = [value for score, value in scored_values if score == best_conflict]
        value = random.choice(best_values)

        current[var] = value
        steps.append(
            f"Step {step}: choose {var}, set {value} "
            f"(conflicts={best_conflict})"
        )

    final_conflicts = csp_total_conflicts(current)
    steps.append(f"Failure: max_steps reached with {final_conflicts} conflict(s)")
    return None, steps, max_steps, final_conflicts


def csp_backtrack(assignment, domains, use_forwardtrack, steps, stats, depth=0):
    if len(assignment) == len(csp_active_variables()):
        steps.append("Solution found")
        return assignment

    var = csp_select_unassigned_variable(assignment)
    steps.append(f"Select {var}")

    for value in domains[var]:
        steps.append(f"Try {var} = {value}")
        stats["nodes"] += 1

        if not csp_is_consistent(var, value, assignment):
            steps.append(f"Reject {var} = {value}")
            continue

        next_assignment = assignment.copy()
        next_assignment[var] = value
        next_domains = csp_copy_domains(domains)
        steps.append(f"Assign {var} = {value}")

        if use_forwardtrack:
            if not csp_color_prune_domains(next_assignment, next_domains, var, value, steps):
                stats["backtracks"] += 1
                steps.append(f"Backtrack from {var} = {value}")
                continue

        result = csp_backtrack(next_assignment, next_domains, use_forwardtrack, steps, stats, depth + 1)
        if result is not None:
            return result

        stats["backtracks"] += 1
        steps.append(f"Backtrack from {var} = {value}")

    return None


def run_csp_search():
    global csp_assignment, csp_current_domains, csp_steps, csp_status
    global csp_nodes_visited, csp_backtracks, csp_steps_scroll_y, csp_info_scroll_y

    csp_steps_scroll_y = 0
    csp_info_scroll_y = 0
    csp_current_domains = csp_copy_domains(csp_active_domains())
    csp_steps = []

    if csp_mode == 2:
        result, domains, steps, arcs_checked, removed_count = csp_ac3()
        csp_assignment = result or {}
        csp_current_domains = domains
        csp_steps = steps
        csp_nodes_visited = arcs_checked
        csp_backtracks = removed_count

        if result is None:
            csp_status = "AC-3 returned failure because a domain became empty."
        else:
            csp_status = f"AC-3 finished with {arcs_checked} arcs checked and {removed_count} value(s) removed."
        return

    if csp_mode == 3:
        result, steps, used_steps, final_conflicts = csp_min_conflicts()
        csp_steps = steps
        csp_nodes_visited = used_steps
        csp_backtracks = final_conflicts

        if result is None:
            csp_assignment = {}
            csp_current_domains = csp_copy_domains(CSP_MAP_DOMAINS)
            csp_status = f"Min-Conflicts returned failure after {used_steps} steps."
            return

        csp_assignment = result
        csp_current_domains = {var: [result[var]] for var in CSP_MAP_VARIABLES}
        csp_status = f"Min-Conflicts solved map coloring in {used_steps} step(s)."
        return

    stats = {"nodes": 0, "backtracks": 0}
    use_forwardtrack = csp_mode == 1

    result = csp_backtrack({}, csp_current_domains, use_forwardtrack, csp_steps, stats)
    csp_nodes_visited = stats["nodes"]
    csp_backtracks = stats["backtracks"]

    if result is None:
        csp_assignment = {}
        csp_status = f"{CSP_MODES[csp_mode]} returned failure."
        return

    csp_assignment = result
    csp_current_domains = {var: [result[var]] for var in csp_active_variables()}
    csp_status = f"{CSP_MODES[csp_mode]} solved CSP with {csp_nodes_visited} tried values."


def reset_csp_result(message=None):
    global csp_assignment, csp_current_domains, csp_steps, csp_status
    global csp_nodes_visited, csp_backtracks, csp_steps_scroll_y, csp_info_scroll_y

    csp_assignment = {}
    csp_current_domains = csp_copy_domains(csp_active_domains())
    csp_steps = []
    csp_nodes_visited = 0
    csp_backtracks = 0
    csp_steps_scroll_y = 0
    csp_info_scroll_y = 0
    csp_status = message or "Ready. Choose a CSP algorithm and press Solve."


def handle_csp_click(mx, my):
    global csp_mode

    if btn_csp_solve.collidepoint(mx, my):
        run_csp_search()
        return

    if btn_csp_reset.collidepoint(mx, my):
        reset_csp_result()
        return

    for idx, rect in enumerate(CSP_MODE_BUTTONS):
        if rect.collidepoint(mx, my):
            csp_mode = idx
            reset_csp_result(f"Selected {CSP_MODES[csp_mode]}.")
            return


def draw_text(text, font, color, pos, anchor="topleft", surface=screen):
    img = font.render(str(text), True, color)
    rect = img.get_rect()
    setattr(rect, anchor, pos)
    surface.blit(img, rect)
    return rect


def draw_text_ellipsis(text, font, color, pos, max_width, anchor="topleft", surface=screen):
    """Draw one line of text and shorten it when the available width is limited."""
    value = str(text)
    suffix = "..."

    if font.size(value)[0] > max_width:
        while value and font.size(value + suffix)[0] > max_width:
            value = value[:-1]
        value = value + suffix

    return draw_text(value, font, color, pos, anchor=anchor, surface=surface)

def draw_shadow(rect, radius=20, alpha=70, offset=(0, 8)):
    shadow = pygame.Surface((rect.w + 24, rect.h + 24), pygame.SRCALPHA)
    shadow_rect = pygame.Rect(12 + offset[0], 12 + offset[1], rect.w, rect.h)
    pygame.draw.rect(shadow, (0, 0, 0, alpha), shadow_rect, border_radius=radius)
    screen.blit(shadow, (rect.x - 12, rect.y - 12))


def draw_card(rect, title=None, subtitle=None):
    draw_shadow(rect, radius=18, alpha=55)
    pygame.draw.rect(screen, CARD, rect, border_radius=18)
    pygame.draw.rect(screen, BORDER, rect, width=1, border_radius=18)
    if title:
        draw_text(title, FONT_HEAD, TEXT, (rect.x + 18, rect.y + 16))
    if subtitle:
        draw_text(subtitle, FONT_SMALL, MUTED, (rect.x + 18, rect.y + 44))


def draw_button(rect, label, bg, hover_bg, text_color=TEXT, active=False):
    mouse_pos = pygame.mouse.get_pos()
    is_hover = rect.collidepoint(mouse_pos)
    color = hover_bg if is_hover or active else bg
    pygame.draw.rect(screen, color, rect, border_radius=12)
    border_color = ACCENT if active else BORDER
    pygame.draw.rect(screen, border_color, rect, width=1, border_radius=12)

    label_font = FONT_BADGE if FONT_BADGE.size(str(label))[0] <= rect.w - 18 else FONT_TINY
    draw_text_ellipsis(
        label,
        label_font,
        text_color,
        rect.center,
        max_width=rect.w - 16,
        anchor="center",
    )
    return is_hover

def tile_rect(i, j):
    return pygame.Rect(
        BOARD_AREA.x + TILE_GAP + j * (TILE_SIZE + TILE_GAP),
        BOARD_AREA.y + TILE_GAP + i * (TILE_SIZE + TILE_GAP),
        TILE_SIZE,
        TILE_SIZE,
    )


def tile_at_pos(pos):
    for i in range(3):
        for j in range(3):
            if tile_rect(i, j).collidepoint(pos):
                return i, j
    return None


def try_move(b, i, j):
    bi, bj = find_zero(b)
    if abs(bi - i) + abs(bj - j) == 1:
        b[bi][bj], b[i][j] = b[i][j], b[bi][bj]
        return True
    return False


def can_swap_any_tile(from_tile, to_tile):
    """Cho kéo một ô số và thả lên bất kỳ ô nào để đổi vị trí trực tiếp."""
    if from_tile is None or to_tile is None:
        return False
    return from_tile != to_tile


def clear_solution_after_manual_move(message):
    global solution_states, moves_used, current_step_index, status_message
    solution_states = []
    moves_used = []
    current_step_index = 0
    status_message = message


def draw_scrollbar(area, scroll_y, content_h, viewport_h):
    max_scroll = max(0, content_h - viewport_h)
    if max_scroll <= 0:
        return
    track = pygame.Rect(area.right - 11, area.y + 54, 5, area.h - 72)
    pygame.draw.rect(screen, (54, 67, 91), track, border_radius=3)
    handle_h = max(28, int(track.h * viewport_h / max(content_h, viewport_h)))
    handle_y = track.y + int((scroll_y / max_scroll) * max(1, track.h - handle_h))
    handle = pygame.Rect(track.x - 2, handle_y, 9, handle_h)
    pygame.draw.rect(screen, ACCENT, handle, border_radius=5)


def draw_mini_board(state, rect, active=False):
    color = CARD_2 if not active else (43, 79, 124)
    pygame.draw.rect(screen, color, rect, border_radius=12)
    pygame.draw.rect(screen, ACCENT if active else BORDER, rect, width=1, border_radius=12)

    gap = 4
    cell = (rect.w - gap * 4) // 3
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            r = pygame.Rect(rect.x + gap + j * (cell + gap), rect.y + gap + i * (cell + gap), cell, cell)
            pygame.draw.rect(screen, EMPTY_TILE if val == 0 else (60, 95, 145), r, border_radius=6)
            if val != 0:
                draw_text(str(val), FONT_SMALL, TEXT, r.center, anchor="center")


def draw_belief_board(state, rect, active=False, label=None):
    pygame.draw.rect(screen, CARD_2 if not active else (43, 79, 124), rect, border_radius=12)
    pygame.draw.rect(screen, ACCENT if active else BORDER, rect, width=1, border_radius=12)

    label_h = 22 if label else 0
    if label:
        draw_text(label, FONT_SMALL, ACCENT if active else MUTED, (rect.x + 8, rect.y + 5))

    inner = pygame.Rect(rect.x + 8, rect.y + label_h + 8, rect.w - 16, rect.h - label_h - 16)
    gap = max(3, min(8, inner.w // 24))
    cell = min((inner.w - gap * 4) // 3, (inner.h - gap * 4) // 3)
    board_w = cell * 3 + gap * 4
    board_h = board_w
    ox = inner.x + (inner.w - board_w) // 2
    oy = inner.y + (inner.h - board_h) // 2
    number_font = FONT_HEAD if cell >= 42 else FONT_SMALL

    for i in range(3):
        for j in range(3):
            val = state[i][j]
            r = pygame.Rect(ox + gap + j * (cell + gap), oy + gap + i * (cell + gap), cell, cell)
            pygame.draw.rect(screen, EMPTY_TILE if val == 0 else (60, 95, 145), r, border_radius=7)
            pygame.draw.rect(screen, (76, 96, 132), r, width=1, border_radius=7)
            if val != 0:
                draw_text(str(val), number_font, TEXT, r.center, anchor="center")


def draw_belief_boards(belief, rect, show_labels=True):
    goals = active_belief_goals()
    states = sorted(belief, key=board_key)
    if not states:
        return

    cols = min(2, len(states))
    rows = (len(states) + cols - 1) // cols
    gap = 14 if show_labels else 8
    max_w = (rect.w - gap * (cols + 1)) // cols
    max_h = (rect.h - gap * (rows + 1)) // rows
    board_size = max(54, min(max_w, max_h))
    total_w = cols * board_size + (cols - 1) * gap
    total_h = rows * board_size + (rows - 1) * gap
    ox = rect.x + (rect.w - total_w) // 2
    oy = rect.y + (rect.h - total_h) // 2

    for idx, state in enumerate(states):
        row = idx // cols
        col = idx % cols
        x = ox + col * (board_size + gap)
        y = oy + row * (board_size + gap)
        label = belief_state_label(state) if show_labels else None
        draw_belief_board(state, pygame.Rect(x, y, board_size, board_size), active=(state in goals), label=label)


def draw_belief_world_section():
    draw_card(BOARD_CARD, "8-Puzzle Belief World", "Both possible boards stay visible until they reach the same goal.")
    display_belief = belief_state_history[-1] if belief_state_history else BELIEF_START
    pygame.draw.rect(screen, BG_2, BELIEF_WORLD_AREA, border_radius=20)
    pygame.draw.rect(screen, BORDER, BELIEF_WORLD_AREA, width=1, border_radius=20)
    draw_belief_boards(display_belief, BELIEF_WORLD_AREA)

    metrics = [
        ("Possible boards", len(display_belief)),
        ("Start boards", len(BELIEF_START)),
        ("Goals", belief_goal_mode + 1),
    ]
    metric_gap = 12
    metric_w = (BOARD_CARD.w - 50 - metric_gap * 2) // 3
    metric_y = BOARD_CARD.y + 598
    for idx, (label, value) in enumerate(metrics):
        metric_rect = pygame.Rect(BOARD_CARD.x + 25 + idx * (metric_w + metric_gap), metric_y, metric_w, 62)
        pygame.draw.rect(screen, BG_2, metric_rect, border_radius=14)
        pygame.draw.rect(screen, BORDER, metric_rect, width=1, border_radius=14)
        draw_text(label, FONT_SMALL, MUTED, (metric_rect.x + 12, metric_rect.y + 9))
        draw_text(str(value), FONT_METRIC, TEXT, (metric_rect.x + 12, metric_rect.y + 31))

    pill = pygame.Rect(BOARD_CARD.x + 25, BOARD_CARD.bottom - 82, BOARD_CARD.w - 50, 56)
    pygame.draw.rect(screen, (24, 31, 46), pill, border_radius=16)
    pygame.draw.rect(screen, BORDER, pill, width=1, border_radius=16)
    status_lower = belief_status.lower()
    if "solved" in status_lower:
        color = SUCCESS
    elif "did not" in status_lower or "no " in status_lower:
        color = WARNING
    else:
        color = MUTED
    draw_text("Status", FONT_SMALL, MUTED, (pill.x + 16, pill.y + 8))
    draw_text_ellipsis(belief_status, FONT_BODY, color, (pill.x + 16, pill.y + 29), max_width=pill.w - 32)


def draw_belief_control_section():
    draw_card(CONTROL_PANEL, "Belief Controls")
    draw_text("Observation Model", FONT_SMALL, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 62))
    for idx, rect in enumerate(BELIEF_MODE_BUTTONS):
        draw_button(rect, BELIEF_MODES[idx], (45, 59, 85), (65, 86, 124), active=(belief_mode == idx))

    draw_text("Goal Test", FONT_SMALL, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 142))
    for idx, rect in enumerate(BELIEF_GOAL_BUTTONS):
        draw_button(rect, BELIEF_GOAL_MODES[idx], (45, 59, 85), (65, 86, 124), active=(belief_goal_mode == idx))

    goals_text = "Every board must be G1" if belief_goal_mode == 0 else "Every board must share G1 or share G2"
    draw_text("Active goals", FONT_TINY, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 226))
    draw_text_ellipsis(goals_text, FONT_HEAD, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 246), CONTROL_PANEL.w - 36)
    draw_button(btn_belief_solve, "Solve", ACCENT_2, ACCENT)
    draw_button(btn_belief_reset, "Reset", (55, 73, 104), (72, 93, 132))


def draw_belief_result_section():
    draw_card(BELIEF_RESULT_PANEL, "Result")
    display_belief = belief_state_history[-1] if belief_state_history else BELIEF_START
    plan_value = len(belief_solution_path) if belief_mode == 0 else len(belief_plan_lines)
    items = [
        ("Mode", BELIEF_MODES[belief_mode]),
        ("Steps", plan_value),
        ("g(n)", belief_solution_cost),
    ]
    box_y = BELIEF_RESULT_PANEL.y + 58
    gap = 10
    widths = [240, 92, 92]
    x = BELIEF_RESULT_PANEL.x + 18
    for idx, ((label, value), width) in enumerate(zip(items, widths)):
        rect = pygame.Rect(x, box_y, width, 54)
        pygame.draw.rect(screen, BG_2, rect, border_radius=12)
        pygame.draw.rect(screen, BORDER, rect, width=1, border_radius=12)
        draw_text(label, FONT_TINY, MUTED, (rect.x + 10, rect.y + 7))
        value_font = FONT_BADGE if idx != 0 else FONT_SMALL
        draw_text_ellipsis(value, value_font, TEXT, (rect.x + 10, rect.y + 28), max_width=rect.w - 20)
        x += width + gap


def draw_belief_plan_section():
    global belief_scroll_y
    draw_card(BELIEF_PLAN_PANEL, "Steps")
    content_x = BELIEF_PLAN_PANEL.x + 18
    content_y = BELIEF_PLAN_PANEL.y + 58
    viewport_w = BELIEF_PLAN_PANEL.w - 42
    viewport_h = BELIEF_PLAN_PANEL.h - 76
    line_h = 24
    rows = len(belief_plan_lines) if belief_plan_lines else 1
    content_h = max(1, rows * line_h)
    max_scroll = max(0, content_h - viewport_h)
    belief_scroll_y = clamp(belief_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)
    if not belief_plan_lines:
        draw_text("No belief steps yet.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        for idx, line in enumerate(belief_plan_lines):
            y = content_y + idx * line_h - belief_scroll_y
            if content_y - line_h <= y <= content_y + viewport_h:
                draw_text_ellipsis(line, FONT_SMALL, TEXT, (content_x, y + 4), max_width=viewport_w - 4)
    screen.set_clip(clip)
    draw_scrollbar(BELIEF_PLAN_PANEL, belief_scroll_y, content_h, viewport_h)


def draw_belief_history_section():
    global belief_tree_scroll_y
    draw_card(BELIEF_TREE_PANEL, "Belief History", "For partial observation, this shows one sensor branch.")
    content_x = BELIEF_TREE_PANEL.x + 18
    content_y = BELIEF_TREE_PANEL.y + 72
    viewport_w = BELIEF_TREE_PANEL.w - 42
    viewport_h = BELIEF_TREE_PANEL.h - 92
    thumb_w = 176
    thumb_h = 126
    gap = 12
    cols = max(1, (viewport_w - gap) // (thumb_w + gap))
    rows = (len(belief_state_history) + cols - 1) // cols if belief_state_history else 1
    content_h = max(1, rows * (thumb_h + 28 + gap))
    max_scroll = max(0, content_h - viewport_h)
    belief_tree_scroll_y = clamp(belief_tree_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)
    for idx, belief in enumerate(belief_state_history):
        row = idx // cols
        col = idx % cols
        x = content_x + col * (thumb_w + gap)
        y = content_y + row * (thumb_h + 28 + gap) - belief_tree_scroll_y
        if y > content_y + viewport_h or y + thumb_h < content_y:
            continue
        mini = pygame.Rect(x, y, thumb_w, thumb_h)
        pygame.draw.rect(screen, CARD_2, mini, border_radius=12)
        pygame.draw.rect(screen, BORDER, mini, width=1, border_radius=12)
        draw_belief_boards(belief, mini.inflate(-10, -10), show_labels=False)
        draw_text(f"B{idx}: {format_belief(belief)}", FONT_SMALL, ACCENT, (x + 2, y + thumb_h + 6))
    screen.set_clip(clip)
    draw_scrollbar(BELIEF_TREE_PANEL, belief_tree_scroll_y, content_h, viewport_h)


def draw_belief_page():
    draw_belief_world_section()
    draw_belief_control_section()
    draw_belief_result_section()
    draw_belief_plan_section()
    draw_belief_history_section()


def draw_csp_graph_section():
    if csp_mode == 0:
        draw_card(
            BOARD_CARD,
            "Backtracking CSP Goal",
            "Variables X1..X9, domain X1..X8={1..8}, X9={0}.",
        )
        graph_area = pygame.Rect(BOARD_CARD.x + 55, BOARD_CARD.y + 112, BOARD_CARD.w - 110, 450)
        tile_size = 126
        tile_gap = 12

        for var in CSP_BACKTRACK_VARIABLES:
            idx = CSP_BACKTRACK_VARIABLES.index(var)
            row = idx // 3
            col = idx % 3
            rect = pygame.Rect(
                graph_area.x + col * (tile_size + tile_gap),
                graph_area.y + row * (tile_size + tile_gap),
                tile_size,
                tile_size,
            )
            value = csp_assignment.get(var)
            target = CSP_BACKTRACK_RULES[var]
            display_text = str(value) if value is not None else "-"
            color = CSP_BACKTRACK_VALUE_COLORS.get(value, (45, 59, 85))
            border_color = ACCENT if value is not None else BORDER

            pygame.draw.rect(screen, color, rect, border_radius=18)
            pygame.draw.rect(screen, border_color, rect, width=3, border_radius=18)
            draw_text(var, FONT_BADGE, TEXT, (rect.x + 14, rect.y + 12))
            draw_text(display_text, FONT_TITLE, TEXT, rect.center, anchor="center")
            rule_color = TEXT if value is not None else (219, 226, 239)
            draw_text(f"rule = {target}", FONT_TINY, rule_color, (rect.centerx, rect.bottom - 22), anchor="center")
    elif csp_mode == 1:
        draw_card(
            BOARD_CARD,
            "Forwardtrack Graph Coloring",
            "Variables A..E, domain {Đỏ, Xanh, Vàng, Trắng}. Adjacent nodes cannot match.",
        )
        graph_area = pygame.Rect(BOARD_CARD.x + 55, BOARD_CARD.y + 90, BOARD_CARD.w - 110, 500)

        for left, right in CSP_COLOR_EDGES:
            x1, y1 = CSP_COLOR_NODE_POS[left]
            x2, y2 = CSP_COLOR_NODE_POS[right]
            p1 = (graph_area.x + x1, graph_area.y + y1)
            p2 = (graph_area.x + x2, graph_area.y + y2)
            pygame.draw.line(screen, (91, 111, 145), p1, p2, width=4)

        for var in CSP_COLOR_VARIABLES:
            x, y = CSP_COLOR_NODE_POS[var]
            center = (graph_area.x + x, graph_area.y + y)
            value = csp_assignment.get(var)
            color = CSP_COLOR_VALUE_COLORS.get(value, (45, 59, 85))
            border_color = ACCENT if value is not None else BORDER
            label_color = (24, 31, 46) if value in ("Vàng", "Trắng") else TEXT
            pygame.draw.circle(screen, color, center, 46)
            pygame.draw.circle(screen, border_color, center, 46, width=4)
            draw_text(var, FONT_HEAD, label_color, (center[0], center[1] - 12), anchor="center")
            draw_text(value if value else "-", FONT_SMALL, label_color, (center[0], center[1] + 18), anchor="center")
    elif csp_mode == 2:
        draw_card(
            BOARD_CARD,
            "AC-3",
            "Domains X={1,2,3}, Y={1,2,3}, Z={2,3}; constraints X<Y and Y<Z.",
        )
        graph_area = pygame.Rect(BOARD_CARD.x + 55, BOARD_CARD.y + 108, BOARD_CARD.w - 110, 430)

        for left, right in CSP_ARC_EDGES:
            x1, y1 = CSP_ARC_NODE_POS[left]
            x2, y2 = CSP_ARC_NODE_POS[right]
            p1 = (graph_area.x + x1, graph_area.y + y1)
            p2 = (graph_area.x + x2, graph_area.y + y2)
            pygame.draw.line(screen, (91, 111, 145), p1, p2, width=4)
            label = "X < Y" if (left, right) == ("X", "Y") else "Y < Z"
            mid = ((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2)
            label_rect = pygame.Rect(mid[0] - 34, mid[1] - 18, 68, 28)
            pygame.draw.rect(screen, (24, 31, 46), label_rect, border_radius=10)
            pygame.draw.rect(screen, BORDER, label_rect, width=1, border_radius=10)
            draw_text(label, FONT_SMALL, ACCENT, label_rect.center, anchor="center")

        for var in CSP_ARC_VARIABLES:
            x, y = CSP_ARC_NODE_POS[var]
            center = (graph_area.x + x, graph_area.y + y)
            domain = csp_current_domains.get(var, CSP_ARC_DOMAINS[var])
            domain_text = "{" + ",".join(str(value) for value in domain) + "}"
            node_rect = pygame.Rect(center[0] - 66, center[1] - 48, 132, 96)

            pygame.draw.rect(screen, (45, 59, 85), node_rect, border_radius=18)
            pygame.draw.rect(screen, ACCENT if len(domain) == 1 else BORDER, node_rect, width=3, border_radius=18)
            draw_text(var, FONT_HEAD, TEXT, (node_rect.centerx, node_rect.y + 18), anchor="center")
            draw_text_ellipsis(domain_text, FONT_BODY, TEXT, (node_rect.centerx, node_rect.y + 58), node_rect.w - 16, anchor="center")
    else:
        draw_card(
            BOARD_CARD,
            "Min-Conflicts Map Coloring",
            "Variables A..D, domain {Đỏ, Xanh, Vàng}. Adjacent regions cannot match.",
        )
        graph_area = pygame.Rect(BOARD_CARD.x + 55, BOARD_CARD.y + 90, BOARD_CARD.w - 110, 500)

        for left, right in CSP_MAP_EDGES:
            x1, y1 = CSP_MAP_NODE_POS[left]
            x2, y2 = CSP_MAP_NODE_POS[right]
            p1 = (graph_area.x + x1, graph_area.y + y1)
            p2 = (graph_area.x + x2, graph_area.y + y2)
            pygame.draw.line(screen, (91, 111, 145), p1, p2, width=4)

        for var in CSP_MAP_VARIABLES:
            x, y = CSP_MAP_NODE_POS[var]
            center = (graph_area.x + x, graph_area.y + y)
            value = csp_assignment.get(var)
            domain = csp_current_domains.get(var, CSP_MAP_DOMAINS[var])
            color = CSP_MAP_VALUE_COLORS.get(value, (45, 59, 85))
            border_color = ACCENT if value is not None else BORDER
            label_color = (24, 31, 46) if value == "Vàng" else TEXT
            detail = value if value else f"{len(domain)} màu"

            pygame.draw.circle(screen, color, center, 48)
            pygame.draw.circle(screen, border_color, center, 48, width=4)
            draw_text(var, FONT_HEAD, label_color, (center[0], center[1] - 14), anchor="center")
            draw_text(detail, FONT_SMALL, label_color, (center[0], center[1] + 18), anchor="center")

    pill = pygame.Rect(BOARD_CARD.x + 25, BOARD_CARD.bottom - 82, BOARD_CARD.w - 50, 56)
    pygame.draw.rect(screen, (24, 31, 46), pill, border_radius=16)
    pygame.draw.rect(screen, BORDER, pill, width=1, border_radius=16)

    status_lower = csp_status.lower()
    color = SUCCESS if ("solved" in status_lower or "finished" in status_lower) else WARNING if "failure" in status_lower else MUTED
    draw_text("Status", FONT_SMALL, MUTED, (pill.x + 16, pill.y + 8))
    draw_text_ellipsis(csp_status, FONT_BODY, color, (pill.x + 16, pill.y + 29), max_width=pill.w - 32)


def draw_csp_control_section():
    draw_card(CONTROL_PANEL, "CSP Controls")
    draw_text("Algorithm", FONT_SMALL, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 62))

    for idx, rect in enumerate(CSP_MODE_BUTTONS):
        draw_button(rect, CSP_MODES[idx], (45, 59, 85), (65, 86, 124), active=(csp_mode == idx))

    draw_text("CSP", FONT_TINY, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 190))
    if csp_mode == 0:
        draw_text("Variables: X1..X9", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 212))
        draw_text("Domain: X1..X8 = {1..8}; X9 = {0}", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 238))
        rule_text = "Rule: X1=1, X2=2, X3=3, X4=4, X5=5, X6=6, X7=7, X8=8, X9=0"
    elif csp_mode == 1:
        draw_text("Variables: A, B, C, D, E", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 212))
        draw_text("Domain: {Đỏ, Xanh, Vàng, Trắng}", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 238))
        rule_text = "Rule: hai biến có cạnh nối nhau thì không được trùng màu"
    elif csp_mode == 2:
        draw_text("Variables: X, Y, Z", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 212))
        draw_text("Domain: X,Y={1,2,3}; Z={2,3}", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 238))
        rule_text = "AC-3: dùng queue arc và xóa giá trị không có support"
    else:
        draw_text("Variables: A, B, C, D", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 212))
        draw_text("Domain: {Đỏ, Xanh, Vàng}", FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 238))
        rule_text = "Min-Conflicts: sửa ngẫu nhiên biến đang conflict"

    draw_text_ellipsis(rule_text, FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 264), CONTROL_PANEL.w - 36)
    draw_button(btn_csp_solve, "Solve", ACCENT_2, ACCENT)
    draw_button(btn_csp_reset, "Reset", (55, 73, 104), (72, 93, 132))


def draw_csp_result_section():
    draw_card(STATS_PANEL, "Result")
    if csp_mode == 2:
        items = [
            ("Algorithm", CSP_MODES[csp_mode]),
            ("Arcs", csp_nodes_visited),
            ("Removed", csp_backtracks),
        ]
    elif csp_mode == 3:
        items = [
            ("Algorithm", CSP_MODES[csp_mode]),
            ("Steps", csp_nodes_visited),
            ("Conflicts", csp_backtracks),
        ]
    else:
        items = [
            ("Algorithm", CSP_MODES[csp_mode]),
            ("Tried", csp_nodes_visited),
            ("Backtrack", csp_backtracks),
        ]
    box_y = STATS_PANEL.y + 58
    gap = 10
    widths = [240, 92, 92]
    x = STATS_PANEL.x + 18

    for idx, ((label, value), width) in enumerate(zip(items, widths)):
        rect = pygame.Rect(x, box_y, width, 54)
        pygame.draw.rect(screen, BG_2, rect, border_radius=12)
        pygame.draw.rect(screen, BORDER, rect, width=1, border_radius=12)
        draw_text(label, FONT_TINY, MUTED, (rect.x + 10, rect.y + 7))
        value_font = FONT_BADGE if idx != 0 else FONT_SMALL
        draw_text_ellipsis(value, value_font, TEXT, (rect.x + 10, rect.y + 28), max_width=rect.w - 20)
        x += width + gap


def draw_csp_steps_section():
    global csp_steps_scroll_y
    draw_card(MOVES_PANEL, "Steps")
    content_x = MOVES_PANEL.x + 18
    content_y = MOVES_PANEL.y + 58
    viewport_w = MOVES_PANEL.w - 42
    viewport_h = MOVES_PANEL.h - 76
    line_h = 24
    rows = len(csp_steps) if csp_steps else 1
    content_h = max(1, rows * line_h)
    max_scroll = max(0, content_h - viewport_h)
    csp_steps_scroll_y = clamp(csp_steps_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)
    if not csp_steps:
        draw_text("No CSP steps yet.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        for idx, line in enumerate(csp_steps):
            y = content_y + idx * line_h - csp_steps_scroll_y
            if content_y - line_h <= y <= content_y + viewport_h:
                draw_text_ellipsis(f"{idx + 1:02d}. {line}", FONT_SMALL, TEXT, (content_x, y + 4), max_width=viewport_w - 4)
    screen.set_clip(clip)
    draw_scrollbar(MOVES_PANEL, csp_steps_scroll_y, content_h, viewport_h)


def draw_csp_info_section():
    global csp_info_scroll_y
    if csp_mode == 0:
        subtitle = "Fixed assignment rules."
    elif csp_mode == 1:
        subtitle = "Forwardtrack graph-coloring constraints."
    elif csp_mode == 2:
        subtitle = "AC-3 queue over X<Y and Y<Z."
    else:
        subtitle = "Min-Conflicts local search over complete assignments."
    draw_card(STATES_PANEL, "CSP Details", subtitle)
    content_x = STATES_PANEL.x + 18
    content_y = STATES_PANEL.y + 72
    viewport_w = STATES_PANEL.w - 42
    viewport_h = STATES_PANEL.h - 92

    if csp_mode == 0:
        variables = CSP_BACKTRACK_VARIABLES
        domains = CSP_BACKTRACK_DOMAINS
        rule_lines = [f"{var} = {CSP_BACKTRACK_RULES[var]}" for var in variables]
        pseudo_lines = [
            "BACKTRACK({}, CSP)",
            "if assignment complete: return assignment",
            "var = select unassigned variable",
            "for value in order-domain-values(var):",
            "  if value matches the rule for var:",
            "    add var=value",
            "    recurse",
            "    remove var=value on failure",
        ]
        section_title = "Rules"
        relation_lines = rule_lines
    elif csp_mode == 1:
        variables = CSP_COLOR_VARIABLES
        domains = CSP_COLOR_DOMAINS
        constraint_lines = [f"{left} != {right}" for left, right in CSP_COLOR_EDGES]
        pseudo_lines = [
            "BACKTRACK({}, CSP)",
            "if assignment complete: return assignment",
            "var = select unassigned variable",
            "for value in order-domain-values(var):",
            "  if no adjacent variable has the same color:",
            "    add var=value",
            "    if Forwardtrack:",
            "      remove value from adjacent unassigned variables",
            "      fail when a domain becomes empty",
            "    recurse",
            "    remove var=value on failure",
        ]
        section_title = "Constraints"
        relation_lines = constraint_lines
    elif csp_mode == 2:
        variables = CSP_ARC_VARIABLES
        domains = CSP_ARC_DOMAINS
        arc_lines = [
            "X domain: {1, 2, 3}",
            "Y domain: {1, 2, 3}",
            "Z domain: {2, 3}",
            "Constraint: X < Y",
            "Constraint: Y < Z",
            "Arcs: X->Y, Y->X, Y->Z, Z->Y",
        ]
        pseudo_lines = [
            "AC-3(csp)",
            "queue = all arcs in csp",
            "while queue is not empty:",
            "  (Xi, Xj) = remove-first(queue)",
            "  if remove-inconsistent-values(Xi, Xj):",
            "    if domain[Xi] is empty: return failure",
            "    for each Xk in neighbors[Xi] except Xj:",
            "      add (Xk, Xi) to queue",
            "return reduced domains",
            "",
            "remove-inconsistent-values(Xi, Xj)",
            "  for each x in domain[Xi]:",
            "    if no y in domain[Xj] satisfies constraint:",
            "      delete x from domain[Xi]",
            "",
            "Expected result",
            "D(X) = {1}",
            "D(Y) = {2}",
            "D(Z) = {3}",
        ]
        section_title = "Problem"
        relation_lines = arc_lines
    else:
        variables = CSP_MAP_VARIABLES
        domains = CSP_MAP_DOMAINS
        constraint_lines = [f"{left} != {right}" for left, right in CSP_MAP_EDGES]
        pseudo_lines = [
            "MIN-CONFLICTS(csp, max_steps)",
            "current = initial complete assignment",
            "for i = 1 to max_steps:",
            "  if current is a solution: return current",
            "  var = random conflicted variable",
            "  value = value minimizing conflicts(var, value)",
            "  set var = value in current",
            "return failure",
        ]
        section_title = "Constraints"
        relation_lines = constraint_lines

    domain_lines = [
        f"{var}: {csp_current_domains.get(var, domains[var])}"
        for var in variables
    ]
    lines = ["Current domains"] + domain_lines + ["", section_title] + relation_lines + ["", "Pseudo-code"] + pseudo_lines

    line_h = 24
    content_h = max(1, len(lines) * line_h)
    max_scroll = max(0, content_h - viewport_h)
    csp_info_scroll_y = clamp(csp_info_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)
    for idx, line in enumerate(lines):
        y = content_y + idx * line_h - csp_info_scroll_y
        if content_y - line_h <= y <= content_y + viewport_h:
            color = ACCENT if line in ("Current domains", "Rules", "Constraints", "Arcs", "Problem", "Pseudo-code") else TEXT
            draw_text_ellipsis(line, FONT_SMALL, color, (content_x, y + 4), max_width=viewport_w - 4)
    screen.set_clip(clip)
    draw_scrollbar(STATES_PANEL, csp_info_scroll_y, content_h, viewport_h)


def draw_csp_page():
    draw_csp_graph_section()
    draw_csp_control_section()
    draw_csp_result_section()
    draw_csp_steps_section()
    draw_csp_info_section()


def draw_andor_world_section():
    draw_card(
        BOARD_CARD,
        "AND-OR Board",
        "Shows the current policy branch found by AND-OR graph search.",
    )
    branch_state = andor_history[-1] if andor_history else ANDOR_START

    pygame.draw.rect(screen, BG_2, BOARD_AREA, border_radius=24)
    pygame.draw.rect(screen, BORDER, BOARD_AREA, width=1, border_radius=24)

    for i in range(3):
        for j in range(3):
            val = branch_state[i][j]
            r = tile_rect(i, j)

            if val == 0:
                pygame.draw.rect(screen, EMPTY_TILE, r, border_radius=18)
                pygame.draw.rect(screen, (46, 58, 82), r, width=1, border_radius=18)
            else:
                draw_shadow(r, radius=18, alpha=35, offset=(0, 4))
                pygame.draw.rect(screen, TILE_BG, r, border_radius=18)
                pygame.draw.rect(screen, (93, 152, 230), r, width=2, border_radius=18)
                draw_text(str(val), FONT_TILE, TEXT, r.center, anchor="center")

    metrics = [
        ("Manhattan h(n)", manhattan_distance(branch_state)),
        ("OR / AND", f"{andor_nodes_visited}/{andor_and_nodes}"),
        ("Depth", andor_policy_depth),
    ]
    metric_gap = 12
    metric_w = (BOARD_CARD.w - 50 - metric_gap * 2) // 3
    metric_y = BOARD_CARD.y + 598

    for idx, (label, value) in enumerate(metrics):
        metric_rect = pygame.Rect(BOARD_CARD.x + 25 + idx * (metric_w + metric_gap), metric_y, metric_w, 62)
        pygame.draw.rect(screen, BG_2, metric_rect, border_radius=14)
        pygame.draw.rect(screen, BORDER, metric_rect, width=1, border_radius=14)
        draw_text(label, FONT_SMALL, MUTED, (metric_rect.x + 12, metric_rect.y + 9))
        draw_text(str(value), FONT_METRIC, TEXT, (metric_rect.x + 12, metric_rect.y + 31))

    pill = pygame.Rect(BOARD_CARD.x + 25, BOARD_CARD.bottom - 82, BOARD_CARD.w - 50, 56)
    pygame.draw.rect(screen, (24, 31, 46), pill, border_radius=16)
    pygame.draw.rect(screen, BORDER, pill, width=1, border_radius=16)
    status_lower = andor_status.lower()
    color = SUCCESS if "found" in status_lower else WARNING if "failed" in status_lower else MUTED
    draw_text("Status", FONT_SMALL, MUTED, (pill.x + 16, pill.y + 8))
    draw_text_ellipsis(andor_status, FONT_BODY, color, (pill.x + 16, pill.y + 29), max_width=pill.w - 32)


def draw_andor_control_section():
    draw_card(CONTROL_PANEL, "Controls")
    draw_button(btn_andor_solve, "Solve", ACCENT_2, ACCENT)
    draw_button(btn_andor_shuffle, "Shuffle", (55, 73, 104), (72, 93, 132))

    draw_text("Algorithm", FONT_SMALL, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 124))
    alg_rect = pygame.Rect(CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 150, CONTROL_PANEL.w - 36, 34)
    draw_button(alg_rect, "AND-OR Graph Search", (45, 59, 85), (65, 86, 124), active=True)

    draw_text("Problem", FONT_TINY, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 214))
    draw_text("8-puzzle conditional search", FONT_HEAD, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 236))
    draw_text("Initial", FONT_TINY, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 282))
    draw_text_ellipsis(format_belief((ANDOR_START,)), FONT_SMALL, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 304), CONTROL_PANEL.w - 36)
    draw_text("Depth limit", FONT_TINY, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 338))
    draw_text(str(ANDOR_MAX_DEPTH), FONT_HEAD, TEXT, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 358))


def draw_andor_result_section():
    draw_card(STATS_PANEL, "Result")
    items = [
        ("Algorithm", "AND-OR"),
        ("OR", andor_nodes_visited),
        ("AND", andor_and_nodes),
    ]
    box_y = STATS_PANEL.y + 58
    gap = 10
    widths = [240, 92, 92]
    x = STATS_PANEL.x + 18

    for idx, ((label, value), width) in enumerate(zip(items, widths)):
        rect = pygame.Rect(x, box_y, width, 54)
        pygame.draw.rect(screen, BG_2, rect, border_radius=12)
        pygame.draw.rect(screen, BORDER, rect, width=1, border_radius=12)
        draw_text(label, FONT_TINY, MUTED, (rect.x + 10, rect.y + 7))
        value_font = FONT_BADGE if idx != 0 else FONT_SMALL
        draw_text_ellipsis(value, value_font, TEXT, (rect.x + 10, rect.y + 28), max_width=rect.w - 20)
        x += width + gap


def draw_andor_steps_section():
    global andor_steps_scroll_y
    draw_card(MOVES_PANEL, "Moves")
    content_x = MOVES_PANEL.x + 18
    content_y = MOVES_PANEL.y + 58
    viewport_w = MOVES_PANEL.w - 42
    viewport_h = MOVES_PANEL.h - 76
    line_h = 24
    rows = len(andor_steps) if andor_steps else 1
    content_h = max(1, rows * line_h)
    max_scroll = max(0, content_h - viewport_h)
    andor_steps_scroll_y = clamp(andor_steps_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)
    if not andor_steps:
        draw_text("No AND-OR steps yet.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        for idx, line in enumerate(andor_steps):
            y = content_y + idx * line_h - andor_steps_scroll_y
            if content_y - line_h <= y <= content_y + viewport_h:
                stripped = line.strip()
                if stripped.startswith("OR:"):
                    color = ACCENT
                elif stripped.startswith("AND:"):
                    color = SUCCESS
                else:
                    color = TEXT
                draw_text_ellipsis(line, FONT_SMALL, color, (content_x, y + 4), max_width=viewport_w - 4)
    screen.set_clip(clip)
    draw_scrollbar(MOVES_PANEL, andor_steps_scroll_y, content_h, viewport_h)


def draw_andor_info_section():
    global andor_info_scroll_y
    draw_card(STATES_PANEL, "State History", "Scroll to review each state in the AND-OR policy branch.")
    content_x = STATES_PANEL.x + 18
    content_y = STATES_PANEL.y + 72
    viewport_w = STATES_PANEL.w - 42
    viewport_h = STATES_PANEL.h - 92

    thumb = 86
    gap = 12
    cols = max(1, (viewport_w - gap) // (thumb + gap))
    rows = (len(andor_history) + cols - 1) // cols if andor_history else 1
    content_h = max(1, rows * (thumb + 26 + gap))
    max_scroll = max(0, content_h - viewport_h)
    andor_info_scroll_y = clamp(andor_info_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)

    if not andor_history:
        draw_text("Press Solve to show states.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        branch_state = andor_history[-1]
        for idx, st in enumerate(andor_history):
            row = idx // cols
            col = idx % cols
            x = content_x + col * (thumb + gap)
            y = content_y + row * (thumb + 26 + gap) - andor_info_scroll_y
            if y > content_y + viewport_h or y + thumb < content_y:
                continue
            active = st == branch_state
            draw_mini_board(st, pygame.Rect(x, y, thumb, thumb), active=active)
            draw_text(f"Step {idx}", FONT_SMALL, ACCENT if active else MUTED, (x + 2, y + thumb + 5))

    screen.set_clip(clip)
    draw_scrollbar(STATES_PANEL, andor_info_scroll_y, content_h, viewport_h)


def draw_andor_page():
    draw_andor_world_section()
    draw_andor_control_section()
    draw_andor_result_section()
    draw_andor_steps_section()
    draw_andor_info_section()


def draw_header():
    titles = [
        "8 Puzzle Search Visualizer",
        "8 Puzzle Belief-State Search",
        "CSP Graph Coloring Visualizer",
        "AND-OR 8-Puzzle Search",
    ]
    subtitles = [
        "Classical, heuristic and local search algorithms",
        "No-observation and partial-observation search over 8-puzzle belief states",
        "Backtracking, Forwardtrack, AC-3 and Min-Conflicts CSP demos",
        "OR chooses an action; AND solves every possible 8-puzzle result",
    ]
    shortcuts = [
        "TAB: switch    SPACE: solve    ESC: quit",
        "TAB: switch    SPACE: solve belief    ESC: quit",
        "TAB: switch    SPACE: solve CSP    ESC: quit",
        "TAB: switch    SPACE: solve AND-OR    ESC: quit",
    ]
    title = titles[active_tab]
    subtitle = subtitles[active_tab]
    shortcut = shortcuts[active_tab]

    draw_text(title, FONT_TITLE, TEXT, (MARGIN, 20))
    subtitle_x = tab_buttons[-1].right + 28
    draw_text_ellipsis(subtitle, FONT_SMALL, MUTED, (subtitle_x, 65), SCREEN_W - subtitle_x - 260)

    tab_strip = pygame.Rect(MARGIN - 4, 52, TAB_W * len(APP_TABS) + 10 * (len(APP_TABS) - 1) + 8, TAB_H + 8)
    pygame.draw.rect(screen, (31, 39, 56), tab_strip, border_radius=14)
    pygame.draw.rect(screen, BORDER, tab_strip, width=1, border_radius=14)

    for idx, rect in enumerate(tab_buttons):
        draw_button(
            rect,
            APP_TABS[idx],
            ACCENT_2 if active_tab == idx else (45, 59, 85),
            ACCENT if active_tab == idx else (65, 86, 124),
            active=(active_tab == idx),
        )

    draw_text(
        shortcut,
        FONT_SMALL,
        MUTED,
        (SCREEN_W - MARGIN, 42),
        anchor="topright",
    )

def draw_board_section():
    draw_card(
        BOARD_CARD,
        "Puzzle Board",
        "Drag a tile onto another cell to create a custom start state.",
    )

    pygame.draw.rect(screen, BG_2, BOARD_AREA, border_radius=24)
    pygame.draw.rect(screen, BORDER, BOARD_AREA, width=1, border_radius=24)

    mouse_pos = pygame.mouse.get_pos()

    for i in range(3):
        for j in range(3):
            val = board[i][j]
            r = tile_rect(i, j)
            hover = r.collidepoint(mouse_pos) and val != 0

            if dragging and drag_tile == (i, j) and val != 0:
                pygame.draw.rect(screen, (28, 37, 54), r, border_radius=18)
                pygame.draw.rect(screen, (65, 82, 112), r, width=1, border_radius=18)
                continue

            if val == 0:
                pygame.draw.rect(screen, EMPTY_TILE, r, border_radius=18)
                pygame.draw.rect(screen, (46, 58, 82), r, width=1, border_radius=18)
            else:
                draw_shadow(r, radius=18, alpha=35, offset=(0, 4))
                pygame.draw.rect(screen, TILE_BG_HOVER if hover else TILE_BG, r, border_radius=18)
                pygame.draw.rect(
                    screen,
                    (130, 184, 255) if hover else (93, 152, 230),
                    r,
                    width=2,
                    border_radius=18,
                )
                draw_text(str(val), FONT_TILE, TEXT, r.center, anchor="center")

    if dragging and drag_value is not None:
        mx, my = drag_pos
        ox, oy = drag_offset
        drag_rect = pygame.Rect(mx - ox, my - oy, TILE_SIZE, TILE_SIZE)
        draw_shadow(drag_rect, radius=18, alpha=70, offset=(0, 8))
        pygame.draw.rect(screen, TILE_BG_HOVER, drag_rect, border_radius=18)
        pygame.draw.rect(screen, ACCENT, drag_rect, width=2, border_radius=18)
        draw_text(str(drag_value), FONT_TILE, TEXT, drag_rect.center, anchor="center")

    # Compact board metrics.
    board_t = board_to_tuple(board)
    metrics = [
        ("Manhattan h(n)", manhattan_distance(board_t)),
        ("Misplaced", misplaced_tiles(board_t)),
        ("Board", "3 x 3"),
    ]
    metric_gap = 12
    metric_w = (BOARD_CARD.w - 50 - metric_gap * 2) // 3
    metric_y = BOARD_CARD.y + 598

    for idx, (label, value) in enumerate(metrics):
        metric_rect = pygame.Rect(
            BOARD_CARD.x + 25 + idx * (metric_w + metric_gap),
            metric_y,
            metric_w,
            62,
        )
        pygame.draw.rect(screen, BG_2, metric_rect, border_radius=14)
        pygame.draw.rect(screen, BORDER, metric_rect, width=1, border_radius=14)
        draw_text(label, FONT_SMALL, MUTED, (metric_rect.x + 12, metric_rect.y + 9))
        draw_text(str(value), FONT_METRIC, TEXT, (metric_rect.x + 12, metric_rect.y + 31))

    pill = pygame.Rect(BOARD_CARD.x + 25, BOARD_CARD.bottom - 82, BOARD_CARD.w - 50, 56)
    pygame.draw.rect(screen, (24, 31, 46), pill, border_radius=16)
    pygame.draw.rect(screen, BORDER, pill, width=1, border_radius=16)

    status_lower = status_message.lower()
    if "solved" in status_lower:
        color = SUCCESS
    elif "not" in status_lower or "did not" in status_lower or "failed" in status_lower:
        color = WARNING
    else:
        color = MUTED

    draw_text("Status", FONT_SMALL, MUTED, (pill.x + 16, pill.y + 8))
    draw_text_ellipsis(
        status_message,
        FONT_BODY,
        color,
        (pill.x + 16, pill.y + 29),
        max_width=pill.w - 32,
    )

def draw_control_section():
    draw_card(CONTROL_PANEL, "Controls")

    draw_button(btn_start, "Start", ACCENT_2, ACCENT)
    draw_button(btn_shuffle, "Shuffle", (55, 73, 104), (72, 93, 132))

    draw_text("Algorithm", FONT_SMALL, MUTED, (CONTROL_PANEL.x + 18, CONTROL_PANEL.y + 124))

    for idx, rect in enumerate(alg_buttons):
        draw_button(
            rect,
            algorithms[idx],
            (45, 59, 85),
            (65, 86, 124),
            active=(selected_alg == idx),
        )

def draw_stats_section():
    draw_card(STATS_PANEL, "Result")

    steps = len(moves_used)
    alg = algorithms[selected_alg]
    solvable = "Yes" if is_solvable(board_to_tuple(board)) else "No"

    box_y = STATS_PANEL.y + 58
    gap = 10
    widths = [240, 92, 92]
    items = [("Algorithm", alg), ("Steps", steps), ("Solvable", solvable)]
    x = STATS_PANEL.x + 18

    for idx, ((label, value), width) in enumerate(zip(items, widths)):
        rect = pygame.Rect(x, box_y, width, 54)
        pygame.draw.rect(screen, BG_2, rect, border_radius=12)
        pygame.draw.rect(screen, BORDER, rect, width=1, border_radius=12)
        draw_text(label, FONT_TINY, MUTED, (rect.x + 10, rect.y + 7))

        value_font = FONT_BADGE if idx != 0 else FONT_SMALL
        draw_text_ellipsis(
            value,
            value_font,
            TEXT,
            (rect.x + 10, rect.y + 28),
            max_width=rect.w - 20,
        )
        x += width + gap

def draw_moves_section():
    global moves_scroll_y
    draw_card(MOVES_PANEL, "Moves")

    content_x = MOVES_PANEL.x + 18
    content_y = MOVES_PANEL.y + 58
    viewport_w = MOVES_PANEL.w - 42
    viewport_h = MOVES_PANEL.h - 76

    cols = 2
    gap_x = 10
    item_h = 34
    row_w = (viewport_w - gap_x) // cols
    rows = (len(moves_used) + cols - 1) // cols if moves_used else 1
    content_h = max(1, rows * item_h)
    max_scroll = max(0, content_h - viewport_h)
    moves_scroll_y = clamp(moves_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)

    if not moves_used:
        draw_text("No moves yet.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        for idx, move in enumerate(moves_used):
            row_idx = idx // cols
            col_idx = idx % cols
            x = content_x + col_idx * (row_w + gap_x)
            y = content_y + row_idx * item_h - moves_scroll_y
            row = pygame.Rect(x, y, row_w, 27)
            pygame.draw.rect(screen, (38, 49, 71), row, border_radius=9)
            draw_text(f"{idx + 1:02d}", FONT_SMALL, ACCENT, (row.x + 10, row.y + 6))
            draw_text(MOVE_LABELS.get(move, move), FONT_SMALL, TEXT, (row.x + 48, row.y + 6))

    screen.set_clip(clip)
    draw_scrollbar(MOVES_PANEL, moves_scroll_y, content_h, viewport_h)

def draw_states_section():
    global state_scroll_y
    draw_card(STATES_PANEL, "State History", "Scroll to review each state in the solution path.")

    content_x = STATES_PANEL.x + 18
    content_y = STATES_PANEL.y + 72
    viewport_w = STATES_PANEL.w - 42
    viewport_h = STATES_PANEL.h - 92

    thumb = 86
    gap = 12
    cols = max(1, (viewport_w - gap) // (thumb + gap))
    rows = (len(solution_states) + cols - 1) // cols if solution_states else 1
    content_h = max(1, rows * (thumb + 26 + gap))
    max_scroll = max(0, content_h - viewport_h)
    state_scroll_y = clamp(state_scroll_y, 0, max_scroll)

    clip = screen.get_clip()
    viewport = pygame.Rect(content_x, content_y, viewport_w, viewport_h)
    screen.set_clip(viewport)

    if not solution_states:
        draw_text("Press Start to show states.", FONT_BODY, MUTED, (content_x, content_y + 8))
    else:
        current_tuple = board_to_tuple(board)
        for idx, st in enumerate(solution_states):
            row = idx // cols
            col = idx % cols
            x = content_x + col * (thumb + gap)
            y = content_y + row * (thumb + 26 + gap) - state_scroll_y
            active = st == current_tuple or idx == current_step_index
            draw_mini_board(st, pygame.Rect(x, y, thumb, thumb), active=active)
            draw_text(f"Step {idx}", FONT_SMALL, ACCENT if active else MUTED, (x + 2, y + thumb + 5))

    screen.set_clip(clip)
    draw_scrollbar(STATES_PANEL, state_scroll_y, content_h, viewport_h)

def redraw():
    screen.fill(BG)
    draw_header()
    if active_tab == 0:
        draw_board_section()
        draw_control_section()
        draw_stats_section()
        draw_moves_section()
        draw_states_section()
    elif active_tab == 1:
        draw_belief_page()
    elif active_tab == 2:
        draw_csp_page()
    else:
        draw_andor_page()
    pygame.display.flip()



def randomize_board():
    nums = list(range(9))
    while True:
        random.shuffle(nums)
        it = iter(nums)
        cand = tuple(tuple(next(it) for _ in range(3)) for _ in range(3))
        if is_solvable(cand) and cand != GOAL:
            return [list(r) for r in cand]


def compute_solution(start_t):
    global solution_path, solution_states, moves_used, status_message, current_step_index

    alg = algorithms[selected_alg]
    if alg == "BFS":
        solution_path = bfs(start_t)
    elif alg == "DFS":
        solution_path = dfs_early_exit(start_t)
    elif alg == "IDDFS":
        solution_path = iddfs(start_t, max_depth=30)
    elif alg == "UCS":
        solution_path = ucs(start_t)
    elif alg == "Greedy":
        solution_path = greedy_search(start_t)
    elif alg == "A*":
        solution_path = astar(start_t)
    elif alg == "IDA*":
        solution_path = ida_star(start_t)
    elif alg == "Hill Climb":
        solution_path = simple_hill_climbing(start_t)
    elif alg == "Steepest Hill Climb":
        solution_path = steepest_hill_climbing(start_t)
    elif alg == "Stochastic Hill Climb":
        solution_path = stochastic_hill_climbing(start_t, max_steps=1000)
    elif alg == "Simulated Annealing":
        solution_path = simulated_annealing(
            start_t,
            initial_temperature=8.0,
            cooling_rate=0.995,
            min_temperature=0.01,
            max_steps=5000,
        )
    elif alg == "Local Beam Search":
        solution_path = local_beam_search(start_t, k=2, max_iterations=1000)
    elif alg == "Random-Restart Hill Climb":
        solution_path = random_restart_hill_climbing(
            start_t,
            max_restarts=100,
            max_random_steps=30,
        )
    else:
        solution_path = None

    current_step_index = 0
    if solution_path is None:
        solution_states = []
        moves_used = []
        if not is_solvable(start_t):
            status_message = "This board is not solvable."
        else:
            status_message = f"{alg} did not find a path. Try BFS."
    else:
        solution_states = [start_t] + [s for (_, s) in solution_path]
        moves_used = [a for (a, _) in solution_path]
        status_message = f"Solved by {alg} in {len(moves_used)} moves."

def animate_solution(path_states, delay=180):
    global board, current_step_index
    for idx, st in enumerate(path_states, start=1):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                raise SystemExit
        board = [list(r) for r in st]
        current_step_index = idx
        redraw()
        pygame.time.wait(delay)


running = True
while running:
    redraw()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.MOUSEWHEEL:
            mx, my = pygame.mouse.get_pos()
            if active_tab == 0:
                if STATES_PANEL.collidepoint(mx, my):
                    state_scroll_y -= event.y * 38
                elif MOVES_PANEL.collidepoint(mx, my):
                    moves_scroll_y -= event.y * 28
            elif active_tab == 1:
                if BELIEF_TREE_PANEL.collidepoint(mx, my):
                    belief_tree_scroll_y -= event.y * 38
                elif BELIEF_PLAN_PANEL.collidepoint(mx, my):
                    belief_scroll_y -= event.y * 28
            elif active_tab == 2:
                if STATES_PANEL.collidepoint(mx, my):
                    csp_info_scroll_y -= event.y * 38
                elif MOVES_PANEL.collidepoint(mx, my):
                    csp_steps_scroll_y -= event.y * 28
            else:
                if STATES_PANEL.collidepoint(mx, my):
                    andor_info_scroll_y -= event.y * 38
                elif MOVES_PANEL.collidepoint(mx, my):
                    andor_steps_scroll_y -= event.y * 28

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            mx, my = event.pos

            clicked_tab = False
            for idx, rect in enumerate(tab_buttons):
                if rect.collidepoint(mx, my):
                    active_tab = idx
                    dragging = False
                    drag_tile = None
                    drag_value = None
                    clicked_tab = True
                    break

            if clicked_tab:
                continue

            if active_tab == 0:
                if btn_start.collidepoint(mx, my):
                    start_t = board_to_tuple(board)
                    compute_solution(start_t)
                    if solution_states:
                        animate_solution(solution_states[1:], delay=180)

                elif btn_shuffle.collidepoint(mx, my):
                    board = randomize_board()
                    orig_board = [row[:] for row in board]
                    clear_solution_after_manual_move("New board generated. Press Start to solve.")

                else:
                    clicked_alg = False
                    for idx, rect in enumerate(alg_buttons):
                        if rect.collidepoint(mx, my):
                            selected_alg = idx
                            status_message = f"Selected {algorithms[selected_alg]}."
                            clicked_alg = True
                            break

                    if not clicked_alg:
                        clicked_tile = tile_at_pos((mx, my))
                        if clicked_tile is not None:
                            i, j = clicked_tile
                            val = board[i][j]
                            if val != 0:
                                dragging = True
                                drag_tile = (i, j)
                                drag_value = val
                                drag_pos = (mx, my)
                                r = tile_rect(i, j)
                                drag_offset = (mx - r.x, my - r.y)
            elif active_tab == 1:
                handle_belief_click(mx, my)
            elif active_tab == 2:
                handle_csp_click(mx, my)
            else:
                handle_andor_click(mx, my)

        elif event.type == pygame.MOUSEMOTION:
            if active_tab == 0 and dragging:
                drag_pos = event.pos

        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            if active_tab == 0 and dragging:
                mx, my = event.pos
                dropped_tile = tile_at_pos((mx, my))

                if can_swap_any_tile(drag_tile, dropped_tile):
                    i, j = drag_tile
                    ti, tj = dropped_tile
                    board[i][j], board[ti][tj] = board[ti][tj], board[i][j]
                    clear_solution_after_manual_move("Tiles swapped freely. Press Start to solve from here.")
                else:
                    status_message = "Drop onto another cell to swap."

                dragging = False
                drag_tile = None
                drag_value = None
                drag_offset = (0, 0)
                drag_pos = (0, 0)

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                running = False
            elif event.key == pygame.K_TAB:
                active_tab = (active_tab + 1) % len(APP_TABS)
                dragging = False
                drag_tile = None
                drag_value = None
            elif event.key == pygame.K_SPACE:
                if active_tab == 0:
                    start_t = board_to_tuple(board)
                    compute_solution(start_t)
                    if solution_states:
                        animate_solution(solution_states[1:], delay=180)
                elif active_tab == 1:
                    run_belief_search()
                elif active_tab == 2:
                    run_csp_search()
                else:
                    run_andor_search()

    clock.tick(60)

pygame.quit()
